# Generador automático — Gestor de conocimiento juventud SDIS

**Qué hace:** Lee la base SIRBE 2025, clasifica los cursos en **4 ejes oficiales** (Bienestar, Cultura, Inclusión social y productiva, Liderazgo y participación), calcula estadísticas y genera el HTML `gestion_conocimiento_juventud_2025.html`.

**Fuente de datos:** `DATOS SIRBE/originales/REQ-SIMI-42642-1292_BasePUA2025_Juventud.xlsx`

**Para regenerar el HTML:** ejecutar todas las celdas en orden.

In [1]:
# Carga de datos
import pandas as pd
import os, re, unicodedata
from datetime import datetime

BASE = os.path.dirname(os.path.abspath("__file__"))
PROYECTO = os.path.dirname(BASE)
# Primero intenta leer la base limpia, si no existe lee la original
RUTA_LIMPIA = os.path.join(PROYECTO, "DATOS SIRBE", "intermedias", "sirbe_2025_limpio.xlsx")
RUTA_ORIGINAL = os.path.join(PROYECTO, "DATOS SIRBE", "originales",
                          "REQ-SIMI-42642-1292_BasePUA2025_Juventud.xlsx")

if os.path.exists(RUTA_LIMPIA):
    df_raw = pd.read_excel(RUTA_LIMPIA)
    print("Leyendo base LIMPIA")
else:
    df_raw = pd.read_excel(RUTA_ORIGINAL, sheet_name="Sheet 1")
    print("Leyendo base ORIGINAL (ejecutar limpieza_sirbe_2025.ipynb primero)")
print(f"Filas totales: {len(df_raw):,.0f}".replace(",", "."))

# Filtrar solo Casas de Juventud (Forjar se trabajará aparte)
df = df_raw[df_raw["SERVICIO"] == "CASAS DE LA JUVENTUD"].copy()
print(f"Filas Casas de Juventud: {len(df):,.0f}".replace(",", "."))
print(f"Columnas: {list(df.columns)}")
df.head(3)

Leyendo base LIMPIA
Filas totales: 39.387
Filas Casas de Juventud: 39.387
Columnas: ['SERVICIO', 'NOMMODALIDAD', 'NOMACTIVIDAD_CURSO', 'ACTUACION_ACTUAL', 'ACTUACION_INTERV_CURSO', 'NOMBRE_CURSO', 'ANIO', 'SEXO', 'EDAD_A_PRIMER_DIA_ANUAL', 'LOCALIDAD_ATENCION', 'NOMTIPPREDIO_ACTUAL', 'GRUPO_ETAREO_1DIA_ANUAL_ACTUA', 'RUV_ACTUAL', 'ETNIA_ACTUAL', 'NOMORIENTACION_ACTUAL', 'SITUACION_DISCAPACIDAD_ACTUAL', 'TIPO_DISCAPACIDAD_ACTUAL', 'IDPERSONA', 'eje', 'NOMBRE_CURSO_LIMPIO']


,SERVICIO,NOMMODALIDAD,NOMACTIVIDAD_CURSO,ACTUACION_ACTUAL,ACTUACION_INTERV_CURSO,NOMBRE_CURSO,ANIO,SEXO,EDAD_A_PRIMER_DIA_ANUAL,LOCALIDAD_ATENCION,NOMTIPPREDIO_ACTUAL,GRUPO_ETAREO_1DIA_ANUAL_ACTUA,RUV_ACTUAL,ETNIA_ACTUAL,NOMORIENTACION_ACTUAL,SITUACION_DISCAPACIDAD_ACTUAL,TIPO_DISCAPACIDAD_ACTUAL,IDPERSONA,eje,NOMBRE_CURSO_LIMPIO
0,CASAS DE LA JUVENTUD,2632 ESTRATEGIA MÓVIL BMT 2938,1502 SOCIALIZACIÓN DE POLÍTICA PÚBLICA DE JUVE...,ATENDIDO DIA (CURSOS),739 POLÍTICA PÚBLICA DE JUVENTUD,SOCIALIZACION DE POLITICA PUBLICA DE JUVENTUD,2025,HOMBRE,17,USME,URBANO SIN DIRECCIÓN,13 Y 17 AÑOS,SI,NINGUNO DE LOS ANTERIORES,HETEROSEXUAL,NO,SIN INFORMACION,214623963961,LIDERAZGO Y PARTICIPACIÓN,Socialización de política pública de juventud
1,CASAS DE LA JUVENTUD,2632 ESTRATEGIA MÓVIL BMT 2938,1498 ORIENTACIÓN SOCIO-OCUPACIONAL DIRIGIDA A ...,ATENDIDO DIA (CURSOS),1005 FORMACIÓN PARA EL PROYECTO DE VIDA,ORIENTACION PSICOSOCIAL,2025,HOMBRE,17,CANDELARIA,URBANO,13 Y 17 AÑOS,NO,NINGUNO DE LOS ANTERIORES,HETEROSEXUAL,NO,SIN INFORMACION,214623985824,INCLUSIÓN SOCIAL Y PRODUCTIVA,Orientación psicosocial
2,CASAS DE LA JUVENTUD,2631 ATENCIÓN INCLUSIVA PARA JÓVENES BMT 2939,1490 ESPACIOS PARA EL DESARROLLO Y FORMACIÓN D...,ATENDIDO DIA (CURSOS),1003 MANEJO ADECUADO DE TIEMPO LIBRE,PROMOCION DDHH A LA PARTICIPACION,2025,MUJER,25,CANDELARIA,URBANO,18 Y 26 AÑOS,NO,NINGUNO DE LOS ANTERIORES,HETEROSEXUAL,NO,SIN INFORMACION,214623052600,CULTURA,"Incidencia, participación y liderazgo"


## Homologación por eje (4 ejes oficiales)

Se usa el código numérico al inicio de `NOMACTIVIDAD_CURSO` para asignar cada registro a uno de los 4 ejes oficiales del servicio de juventud.

In [2]:
# Homologación por eje (4 ejes oficiales)
# Fuente: PPT "Primera reunión del servicio" + script homologacion_sirbe

HOMOLOGACION_EJES = {
    1485: "BIENESTAR", 1486: "BIENESTAR", 1487: "BIENESTAR",
    1488: "BIENESTAR", 1599: "BIENESTAR", 511: "BIENESTAR",
    1489: "CULTURA", 1490: "CULTURA", 1207: "CULTURA", 1491: "CULTURA",
    1496: "INCLUSIÓN SOCIAL Y PRODUCTIVA", 1497: "INCLUSIÓN SOCIAL Y PRODUCTIVA",
    1498: "INCLUSIÓN SOCIAL Y PRODUCTIVA", 1499: "INCLUSIÓN SOCIAL Y PRODUCTIVA",
    1492: "LIDERAZGO Y PARTICIPACIÓN", 1493: "LIDERAZGO Y PARTICIPACIÓN",
    1494: "LIDERAZGO Y PARTICIPACIÓN", 1495: "LIDERAZGO Y PARTICIPACIÓN",
    1502: "LIDERAZGO Y PARTICIPACIÓN", 1503: "LIDERAZGO Y PARTICIPACIÓN",
    1504: "LIDERAZGO Y PARTICIPACIÓN", 1505: "LIDERAZGO Y PARTICIPACIÓN",
}
df["cod_actividad"] = pd.to_numeric(
    df["NOMACTIVIDAD_CURSO"].astype(str).str[:4], errors="coerce"
)
df["eje"] = df["cod_actividad"].map(HOMOLOGACION_EJES).fillna("SIN CLASIFICAR")

# Distribución por eje
print("Distribución por eje:")
for eje, n in df["eje"].value_counts().items():
    print(f"  {eje:<40s} {n:>6}  ({n/len(df)*100:.1f}%)")

Distribución por eje:
  BIENESTAR                                 22748  (57.8%)
  INCLUSIÓN SOCIAL Y PRODUCTIVA              8987  (22.8%)
  CULTURA                                    5201  (13.2%)
  LIDERAZGO Y PARTICIPACIÓN                  2284  (5.8%)
  SIN CLASIFICAR                              167  (0.4%)


## Subtemas por eje

Cada eje tiene subtemas derivados de `NOMBRE_CURSO` mediante reglas de regex. Esto permite desglosar la oferta dentro de cada eje.

In [3]:
# Subtemas por eje (basados en NOMBRE_CURSO)

SUBTEMAS = {
    "BIENESTAR": [
        ("Prevención de consumo de SPA", [r"PREVENCION.*(?:CONSUMO|SPA|SUSTANCIA)", r"PREVENCI.N.*(?:CONSUMO|SPA)", r"PREVENSION.*CONSUMO", r"POREVENCION", r"\bSPA\b", r"SUSTANCIAS PSICO", r"CONSUMO.*SPA", r"CUIDADO FRENTE AL CONSUMO", r"CLOSET PSICOACTIVO", r"CON SUMO CUIDADO", r"PATRONES DE CONSUMO"]),
        ("Prevención de violencias basadas en género", [r"VIOLENCIA", r"VBG", r"PVBG", r"DVBG", r"ESTEREOTIPOS DE GENERO", r"MASCULINIDADES", r"MITOS DEL AMOR ROMANTICO", r"25.?N"]),
        ("Derechos sexuales y reproductivos", [r"DERECHO.? SEXUAL", r"REPRODUCTIV", r"DSDR", r"DSYR", r"DRYDR", r"DSYDR", r"DDSS", r"DS\.?R", r"DDRRYRR", r"ANATOMIA", r"ANTICONCEP", r"SEXUALIDAD", r"MENSTR", r"MI CUERPO MI", r"CONSENTIMIENTO SEXUAL", r"SEMANA ANDINA", r"PROMOCION DSR"]),
        ("Salud mental", [r"SALUD MENTAL", r"AUTOCUIDADO", r"BIENESTAR SALUD", r"SUICIDIO", r"CONDUCTA SUICIDA", r"MINDFUL", r"RESPIRO", r"CUIDADOR", r"JORNADA DE BIENESTAR", r"INTERCAMBIO DE SABERES", r"SALAS DE ESCUCHA", r"SALAS DE MEDITACION", r"ARTE DE SENTIR", r"DIMENSIONES DE LA SALUD"]),
        ("Gestión emocional", [r"GESTION EMOCIONAL", r"REGULACION EMOCIONAL", r"RESOLUCION DE CONFLICTOS", r"COMUNICACION ASERTIVA", r"INTELIGENCIA EMOCIONAL", r"ESTABLECIMIENTO DE LIMITES", r"CUIDAR PARA RESOLVER", r"CIUDAR PARA RESOLVER", r"RECONOCIMIENTO DE EMOCIONES", r"CONECTANDO EMOCIONES"]),
        ("Orientación psicosocial", [r"ORIENTACION.*PSICOSOCIAL", r"PSICOSOCIAL", r"HABILIDADES.*VIDA"]),
    ],
    "CULTURA": [
        ("Artes y formación artística", [r"ARTE", r"PLASTICAS", r"ESCENICAS", r"MUSICALES", r"URBANAS", r"AUDIOVISUAL", r"FOTOGRAFIA", r"GRAFITI", r"GRAFFIT", r"GRAFFITI", r"MURALISMO", r"FORMACION ARTISTICA", r"PRACTICAS ARTISTICAS", r"CLASES ARTISTICAS", r"CIRCO", r"CIRCENSES", r"DANZA", r"TEATRO", r"MUSICA", r"ROCK", r"GUITARRA", r"PIANO", r"BATERIA", r"MALABARISMO", r"SERIGRAFIA", r"CINE", r"PODCAST", r"GRABACION", r"ENSAYAD", r"SALAS DE ENSAYO", r"IDARTES", r"STOP MOTION", r"SAFARI FOTOGRAFICO"]),
        ("Uso de espacios y gestión cultural", [r"USO DE ESPACIOS", r"GESTION CULTURAL", r"GESTION CUTURAL"]),
        ("Deportes y recreación", [r"DEPORTE", r"DEPORTIV", r"CAPOEIRA", r"BOXEO", r"FUTBOL", r"VOLEIBOL", r"ULTIMATE", r"PING PONG", r"SEMILLERO", r"TORNEO", r"ENCUENTRO DEPORTIVO"]),
        ("Eventos y semanas temáticas", [r"SEMANA DE LA JUVENTUD", r"SEMANA DE JUVENTUD", r"SEMANA LOCAL", r"HALLOWEEN", r"CONCIERTO", r"REINADO", r"BIZ NATION"]),
        ("Tiempo libre e idiomas", [r"TIEMPO LIBRE", r"APROVECHAMIENTO", r"IDIOMAS", r"INGLES"]),
    ],
    "INCLUSIÓN SOCIAL Y PRODUCTIVA": [
        ("Empleo y empleabilidad", [r"EMPLEO", r"EMPLEABILIDAD", r"EMBLEABILIDAD", r"INTERMEDIACION LABORAL", r"HOJA DE VIDA", r"FERIA.*EMPLEA", r"INCLUSION LABORAL", r"FORMACION DE EMPLEND", r"CRECIENDO JUNTOS", r"DISRUPTIA"]),
        ("Emprendimiento", [r"EMPRENDIMIENTO", r"EMPRENDIMIENT", r"INNOVERS", r"ISYP", r"GENERACION DE INGRESOS"]),
        ("Educación financiera", [r"EDUCACI.N FINANCIERA", r"FINANZ", r"AHORRO", r"PRESUPUESTO", r"COSTO INVISIBLE"]),
        ("Proyecto de vida y educación flexible", [r"PROYECTO DE VIDA", r"EDUCACION FLEXIBLE", r"EDUCACION RUTA", r"MODELO.*EDUCACION FLEXIBLE", r"RUTAS.*EDUCACION", r"FORMACION RUTAS", r"FORMACION RUTA", r"FORMACION PARA EL PROYECTO", r"ACCESO A LA EDUCACI", r"ACCESO A EDUCACION", r"LECTO.?ESCRITURA", r"PRE.?ICFES", r"PREICFES"]),
        ("Orientación socio-ocupacional", [r"ORIENTACION SOCIO", r"ORIENTACION.*INCLUSION", r"INCLUSION SOCIAL.*PRODUCTIV"]),
        ("Habilidades TIC", [r"HABILIDADES INFORMATICA", r"OFERTA TIC", r"SALA.? TIC", r"FORTALECIMIENTO.*HABILIDADES"]),
    ],
    "LIDERAZGO Y PARTICIPACIÓN": [
        ("Política pública de juventud", [r"POLITICA PUBLICA", r"POLITIC.? PUBLICA", r"PLITICA PUBLICA", r"SICIALIZACION DE POLITICA", r"SOCIALIZACION.*POLITICA", r"SOCIALIZACION POLITIC"]),
        ("Liderazgo juvenil", [r"LIDERAZGO", r"LIDEREAZGO", r"ORGANIZACION JUVENIL", r"PARTICIPACION.*JUVENIL", r"INCIDENCIA"]),
        ("Asesoría jurídica y participación", [r"ASESORIA JURIDICA", r"ASESPROA JURIDICA", r"SESORIA JURIDICA", r"JURIDICA"]),
        ("Derechos humanos y ciudadanía", [r"DERECHOS HUMANOS", r"PROMOCION.*DDHH", r"CONTROL SOCIAL"]),
        ("Voluntariado y diálogos", [r"VOLUNTARIADO", r"DIALOGOS INTERGENERACIONAL", r"ESCENARIOS DE DIALOGO"]),
    ],
}

def subtematizar(nombre, eje):
    """Asigna un subtema dentro del eje correspondiente."""
    if pd.isna(nombre) or eje not in SUBTEMAS:
        return "Otros"
    texto = nombre.upper()
    for subtema, patrones in SUBTEMAS[eje]:
        for p in patrones:
            if re.search(p, texto):
                return subtema
    return "Otros"

df["subtema"] = df.apply(lambda r: subtematizar(r["NOMBRE_CURSO"], r["eje"]), axis=1)

# Revisar subtemas por eje
for eje in ["BIENESTAR", "CULTURA", "INCLUSIÓN SOCIAL Y PRODUCTIVA", "LIDERAZGO Y PARTICIPACIÓN"]:
    sub = df[df["eje"] == eje]
    print(f"\n{eje}:")
    for st, n in sub["subtema"].value_counts().items():
        print(f"    {st:<45s} {n:>5}  ({n/len(sub)*100:.1f}%)")


BIENESTAR:
    Derechos sexuales y reproductivos              7892  (34.7%)
    Salud mental                                   5454  (24.0%)
    Prevención de consumo de SPA                   4253  (18.7%)
    Prevención de violencias basadas en género     3985  (17.5%)
    Gestión emocional                               670  (2.9%)
    Otros                                           484  (2.1%)
    Orientación psicosocial                          10  (0.0%)

CULTURA:
    Artes y formación artística                    3439  (66.1%)
    Uso de espacios y gestión cultural              689  (13.2%)
    Eventos y semanas temáticas                     426  (8.2%)
    Deportes y recreación                           291  (5.6%)
    Otros                                           252  (4.8%)
    Tiempo libre e idiomas                          104  (2.0%)

INCLUSIÓN SOCIAL Y PRODUCTIVA:
    Empleo y empleabilidad                         2054  (22.9%)
    Emprendimiento                         

In [4]:
# Revisar registros sin clasificar (para ajustar reglas si es necesario)
sin_clasificar = df[df["eje"] == "SIN CLASIFICAR"]
print(f"Registros SIN CLASIFICAR: {len(sin_clasificar)} ({len(sin_clasificar)/len(df)*100:.1f}%)")
if len(sin_clasificar) > 0:
    print("\nCursos sin clasificar (top 20):")
    for curso, n in sin_clasificar["NOMBRE_CURSO"].value_counts().head(20).items():
        cod = sin_clasificar[sin_clasificar["NOMBRE_CURSO"] == curso]["cod_actividad"].iloc[0]
        print(f"  cod={int(cod) if pd.notna(cod) else '?':>5}  {n:>4}x  {curso}")

# Registros con subtema "Otros" dentro de ejes clasificados
otros = df[(df["eje"] != "SIN CLASIFICAR") & (df["subtema"] == "Otros")]
print(f"\nRegistros con subtema 'Otros' (dentro de ejes): {len(otros)} ({len(otros)/len(df)*100:.1f}%)")
if len(otros) > 0:
    print("\nCursos en 'Otros' (top 15):")
    for curso, n in otros["NOMBRE_CURSO"].value_counts().head(15).items():
        eje = otros[otros["NOMBRE_CURSO"] == curso]["eje"].iloc[0]
        print(f"  [{eje[:12]:<12}] {n:>4}x  {curso}")

Registros SIN CLASIFICAR: 167 (0.4%)

Cursos sin clasificar (top 20):
  cod=    ?   102x  EDUCACIN FINANCIERA
  cod=    ?    24x  TALLER DERECHOS SEXUALES Y REPRODUCTIVOS
  cod=    ?    20x  TALLER PREVENCION DE CONSUMO DE SUSUTANCIAS PSICOACTIVAS
  cod=    ?    11x  EDUCACION FINANCIERA
  cod=    ?    10x  TALLER PREVENCION DE VIOLENCIAS BASADAS EN GENERO

Registros con subtema 'Otros' (dentro de ejes): 1884 (4.8%)

Cursos en 'Otros' (top 15):
  [INCLUSIÓN SO]  403x  EM TALLER DE FORMACIÓN EN EDUCACIÓN CONTINUA
  [BIENESTAR   ]  119x  EM TALLER DE GESTION DE EMOCIONAL
  [INCLUSIÓN SO]  109x  IDIOMAS
  [INCLUSIÓN SO]   76x  FORMACION METODOLOGIAS DE CUIDADO JCO
  [BIENESTAR   ]   72x  BI LIMITES CLAROS Y BULLYING
  [INCLUSIÓN SO]   66x  EM TALLER DE EDUCACION CONTINUA
  [INCLUSIÓN SO]   58x  FORMACION METODOLOGIAS DEL CUIDADO JCO
  [BIENESTAR   ]   51x  EL COSTO INVISIBLE
  [INCLUSIÓN SO]   51x  FORMACION TENENCIA RESPONSABLE DE MASCOTAS
  [BIENESTAR   ]   50x  SOCIALIZACION DE PREVENC

## Normalización de nombres de curso

Agrupa variaciones del mismo curso (prefijos BI/EM, tildes, errores ortográficos) en un nombre limpio para el top 10 de cada eje.

In [5]:
# Normalización de nombres de curso
def normalizar_curso(nombre):
    """Agrupa variaciones del mismo curso (prefijos BI/EM, tildes, errores)."""
    if pd.isna(nombre):
        return "SIN NOMBRE"
    t = nombre.upper().strip()
    t = unicodedata.normalize("NFD", t)
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    # Quitar prefijos
    for p in [r"^BI\s+TALLER\s+(?:DE\s+|EN\s+)?", r"^EM\s+TALLER\s+(?:DE\s+|EN\s+)?",
              r"^TALLER\s+(?:DE\s+|EN\s+|SOBRE\s+)?", r"^TALLERES\s+(?:DE\s+|EN\s+|INFORMATIVOS\s+DE\s+)?",
              r"^BI\s+", r"^EM\s+", r"^BIENESTAR\s+", r"^CULTURA\s+",
              r"^FORMACION\s+EN\s+", r"^SOCIALIZACION\s+DE\s+",
              r"^SOCIALIZACION\s+", r"^JORNADA\s+DE\s+",
              r"^PROMOCION\s+Y?\s*", r"^INCLUSION\s+SOCIAL\s+"]:
        t = re.sub(p, "", t)
    # Normalizar variaciones conocidas
    for patron, reemplazo in [
        # --- DSDR ---
        (r"DERECHOS SEXUALES Y DERECHOS REPRODUCTIVOS", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUIALES DERECHOS REPRODUCTIVOS", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES\s+DERECHOS REPRODUCTIVOS", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DSYDR.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS\s+SEMANA.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS\s+DECIDE.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS\s+AUTOCUIDADO.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS\s+MITOS.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS\s+GRATIFERIA", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS\s+FERIA.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS\s+TALLER.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"INFORMATIVO EN DERECHOS SEXUALES Y REPRODUCTIVOS", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"LABORATORIO DERECHOS SEXUALES Y REPRODUCTIVOS", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"INTERCAMBIO DE SABERES TEMATICA DSYR", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DECIME COMO CUANDO.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"CONOCE TUS DERECHOS.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"^DSDR.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"MURAL SOBRE CONSENTIMIENTO SEXUAL", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"CINE COMUNITARIO 8M BIENESTAR DS Y DR", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"FERIA PROMOCION DSR.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y REPRODUCTIVOS SEMANA DE JUVENTUD", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        (r"DERECHOS SEXUALES Y DERECHOS REPRODUCTIVOS.*", "DERECHOS SEXUALES Y REPRODUCTIVOS"),
        # --- Prevención consumo SPA ---
        (r"PREVENCI.?N DE CONSUMO DE SUSTANCIAS?\s?PSICO?ACTIVAS?\s?(?:SPA)?", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION CONSUMO DE?\s?(?:SUSTANCIAS?\s?)?(?:PSICO?ACTIVAS?\s?)?(?:SPA)?", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE CONSUMO?DE?\s?SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DEL? CONSUMO DE SUSTANCIAS PSICOACTIVAS SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE SUSTANCIAS PSICOACTIVAS SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE CONSUMODE SUSTANCIAS PSICOACTIVAS", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION Y CONSUMO DE SUSTANCIAS PSICOACTIVAS SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE CONSUMO DE SUSTACIAS PSICOACTIVAS", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE CONSUMO DE SUTANCIAS PSICOACTIVAS", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENSION DE? CONSUMO SUSTANCIAS PSICOACTIVAS SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"POREVENCION DE CONSUMO DE SUSTANCIAS PSICOACTIVAS", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE SPA.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION SPA.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"SPA GESTION EMOCIONAL.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DECIDE COMO CUANDO.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"SEMANA DE PREVENCION DEL CONSUMO DE SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"SPA MITOS DEL CONSUMO", "PREVENCION DE CONSUMO DE SPA"),
        (r"CUIDADO FRENTE AL CONSUMO.*SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION CONSUMOI DE SUSTACIOAS.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DEL CONSUMO DE SPA.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION CONSUMO SPA.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCCION DEL CONSUMO DE SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION EN CONSUMO.*SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE CONSUMO DE SUSTANCIAS PSICOACTIVAS SPA", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE CONSUMO DE SUSTANCIAS PSICOACTIVAS$", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION DE CONSUMO DE SUSTANCIA PSICOACTIVAS.*", "PREVENCION DE CONSUMO DE SPA"),
        (r"PREVENCION CONSUMO SUSTANCIAS.*", "PREVENCION DE CONSUMO DE SPA"),
        # --- Prevención VBG ---
        (r"PREVENCI.?N DE VIOLENCIAS?\s?BASADA?S?\s?EN\s?G.?NERO?S?.*", "PREVENCION DE VBG"),
        (r"PREVENCION (?:DE |EN )?VIVENCIAS BASADAS EN GENERO", "PREVENCION DE VBG"),
        (r"PREVENCION (?:EN |DE )?VIOLENCIAS? BASADA?S? EN GENERO?S?.*", "PREVENCION DE VBG"),
        (r"PREVENCION DEVIOLENCIAS BASADAS EN GENERO", "PREVENCION DE VBG"),
        (r"PREVENCIN DE VIOLENCIAS BASADAS EN GENERO", "PREVENCION DE VBG"),
        (r"PREVENCION DE VIOLENCIA BASADOS EN GENERO", "PREVENCION DE VBG"),
        (r"PREVENCION DE VIOLENCIAS BASADO EN GENERO", "PREVENCION DE VBG"),
        (r"PREVENCION DE VIOLENCIAS\b", "PREVENCION DE VBG"),
        (r"PREVENCION DE VBG\b", "PREVENCION DE VBG"),
        (r"PVBG.*", "PREVENCION DE VBG"),
        (r"DVBG.*", "PREVENCION DE VBG"),
        (r"POR ELLAS POR TODAS POR NOSOTRAS", "PREVENCION DE VBG"),
        (r"PREVENCION DE VIOLENCIAS DIGITALES", "PREVENCION DE VBG"),
        (r"OFERTA Y PROYECTO ALCALDIA PVBG", "PREVENCION DE VBG"),
        (r"PREVENCION DE VIOLENCIAS ESTEREOTIPOS DE GENERO", "PREVENCION DE VBG"),
        (r"CINE FORO PVBG", "PREVENCION DE VBG"),
        (r"TALLERES DE PREVENCIN DE VIOLENCIAS", "PREVENCION DE VBG"),
        (r"^PREVENCIN DE VIOLENCIAS$", "PREVENCION DE VBG"),
        (r"^PREVENCION DE VIOLENCIAS$", "PREVENCION DE VBG"),
        # --- Salud mental ---
        (r"SALUD MENTAL\s+CONVERSATORIO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+GESTION EMOCIONAL", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+AUTOCUIDADO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+BIENESTAR.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+TRANSFORMACI?O?N CULTURAL.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+JORNADA.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+PREVENCION.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+TOMA DE DECISIONES", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+RECONOCIMIENTO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+DIMENSIONES.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+ESTABLECIMIENTO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+IDENTIDAD.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+CONECTANDO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+CUERPOS.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+CUIDADO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+CICLO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+SERVICIO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+PODCAST.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+TEJIENDO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+MANDALAS", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+EL ARTE.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+SEMANA.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+CONCIERTO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+FERIA.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+GRIPO.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+JORNADAS.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+TALLER\b.*", "SALUD MENTAL"),
        (r"SALUD MENTAL\s+GRUPO FOCAL", "SALUD MENTAL"),
        (r"^GESTION DE EMOCIONAL$", "GESTION EMOCIONAL Y RESOLUCION DE CONFLICTOS"),
        (r"^CONVERSATORIO SALUD MENTAL.*", "SALUD MENTAL"),
        (r"^BIENESTAR E IDENTIDAD$", "SALUD MENTAL"),
        # --- Cultura: uso de espacios ---
        (r"USO DE ESPACI?OS?\s?PARA\s?(?:EL\s)?DESARROLL?O?S?\s?(?:DE\s)?(?:LAS?\s)?PR[AE]CTICAS?\s?(?:ARTISTICAS?|JUVENILES)(?:\s?CDJ)?.*", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO (?:DE ESPACOS|PARA EL DESARROLLO DE PRACTICAS ARTISTICAS).*", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USOS? DE ESPACIOS PARA DESARROLLO PREACTICAS ARTISTICAS", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS LIBRES DE PRACTICA ARTISTICA", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO ADECUADO DE ESPACIOS PARA EL DESARROLLO DE PRACTICAS ART.*", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS CDJ", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS CONCURSO.*", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS DE CDJ.*", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS SEMANA.*", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS PARA DESARROLLO DE PRACTICAS ARTISTICAS.*", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS PARA DESARROLLOS DE PRACTICAS ARTISTICAS", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS PARA DESARROLOS DE PRACTICAS ARTISTICAS", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        (r"USO DE ESPACIOS POLITICA PUBLICA.*", "POLITICA PUBLICA DE JUVENTUD"),
        (r"USO DE ESPACIOS PARA DESARROLLO DE PRACTICAS ARTISTICAS", "USO DE ESPACIOS PARA PRACTICAS ARTISTICAS"),
        # --- Inclusión: empleo ---
        (r"^EMPLEO\s+SEMANA.*", "EMPLEO"),
        (r"^EMPLEO\s+FERIA.*", "EMPLEO"),
        (r"^EMPLEO\s+FORMACION.*", "EMPLEO"),
        (r"^EMPLEO\s+TALLER.*", "EMPLEO"),
        (r"^EMPLEO\s+ANDI.*", "EMPLEO"),
        (r"^EMPLEO\s+HERRAMIENTAS.*", "EMPLEO"),
        (r"SEMANA DE LA JUVENTUD EMPLEO", "EMPLEO"),
        (r"SEMANA DE LA JUVENTUD FERIA.*", "EMPLEO"),
        (r"^EMPLEOSEMANA.*", "EMPLEO"),
        (r"^EMPLEO_SEMANA.*", "EMPLEO"),
        (r"SERVICIOS\s+DE INTERMEDIACION LABORAL", "EMPLEO"),
        (r"SERVICIO DE INTERMEDIACION LABORAL", "EMPLEO"),
        (r"FERIA EMPLEABILIDAD EMPLEO", "EMPLEO"),
        (r"ACCIONES FORMACION EN EMPLEABILIDAD", "EMPLEO"),
        (r"FORMACION DE EMPLEABILIDAD.*", "EMPLEO"),
        (r"EMPLEABILIDAD\b", "EMPLEO"),
        (r"^EMPLO$", "EMPLEO"),
        (r"INCLUSION SOCIAL Y PRODUCTIVA EMPLEO", "EMPLEO"),
        (r"^ACCIONES PARA LA INCLUSION LABORAL.*", "EMPLEO"),
        # --- Inclusión: emprendimiento ---
        (r"EMPRENDIMIENTO.INNOVERS", "EMPRENDIMIENTO INNOVERS"),
        (r"EMPRENDIMIENTP", "EMPRENDIMIENTO"),
        (r"^EMPRENDIMIENTO\s+SEMANA.*", "EMPRENDIMIENTO"),
        (r"^EMPRENDIMIENTO\s+FERIA.*", "EMPRENDIMIENTO"),
        (r"^EMPRENDIMIENTO\s+FERIAS.*", "EMPRENDIMIENTO"),
        (r"^EMPRENDIMIENTO\s+TALLERES.*", "EMPRENDIMIENTO"),
        (r"^EMPRENDIMIENTO\s+TALLER.*", "EMPRENDIMIENTO"),
        (r"^EMPRENDIMIENTO\s+ISYP", "EMPRENDIMIENTO"),
        (r"^EMPRENDIMIENTO\s+FORMACION.*", "EMPRENDIMIENTO"),
        (r"ISYP EMPRENDIMIENTO", "EMPRENDIMIENTO"),
        (r"Y PRODUCTIVA EMPRENDIMIENTO", "EMPRENDIMIENTO"),
        (r"INCLUSION PRODUCTIVA EMPRENDIMIENTO", "EMPRENDIMIENTO"),
        # --- Inclusión: proyecto de vida ---
        (r"^DEL PROYECTO DE VIDA", "PROYECTO DE VIDA"),
        (r"PARA EL PROYECTO DE VIDA.*", "PROYECTO DE VIDA"),
        (r"PROYECTO DE VIDA\b", "PROYECTO DE VIDA"),
        # --- Educación financiera ---
        (r"EDUCACI.N FINANCIERA.*", "EDUCACION FINANCIERA"),
        (r"ORIENTACION SOCIO.?OCUPACIONAL.*", "ORIENTACION SOCIO-OCUPACIONAL"),
        # --- Habilidades TIC ---
        (r"HABILIDADES INFORMATIC.?S?\s?OFERTA\s?TICS?", "HABILIDADES TIC"),
        (r"HABILIDADES INFORMATIOCAS OFERTA TIC", "HABILIDADES TIC"),
        (r"OFERTA TIC HABILIDADES INFORMATICAS", "HABILIDADES TIC"),
        (r"FORTALECIMIENTO DE HABILIDADES.*TIC", "HABILIDADES TIC"),
        (r"FORTALECIMIENTO HABILIDADES SALA TIC", "HABILIDADES TIC"),
        (r"SALA TIC.*", "HABILIDADES TIC"),
        (r"HABILIDADES INFORMATICA.*", "HABILIDADES TIC"),
        # --- Educación flexible ---
        (r"(?:EDUCACION\s+)?(?:FORMACION\s+)?RUTAS?\s?DE?\s?MODELO\s?DE?\s?EDUCACI.?N FLEXIBLE", "EDUCACION FLEXIBLE"),
        (r"FORMACION\s+RUTA\s+(?:DE\s+)?MODELO\s+DE\s+EDUCACION FLEXIBLE", "EDUCACION FLEXIBLE"),
        (r"MODELO\s?FORMACION\s?RUTA\s?DE\s?EDUCACION FLEXIBLE", "EDUCACION FLEXIBLE"),
        (r"EDUCACION RUTA DE MODELO DE EDUCACION FLEXIBLE", "EDUCACION FLEXIBLE"),
        (r"FORMACIN RUTAS MODELO DE EDUCACION FLEXIBLE", "EDUCACION FLEXIBLE"),
        (r"RUTAS DE MODELO DE EDUCACION FLEXIBLE", "EDUCACION FLEXIBLE"),
        (r"RUTA DE MODELO DE EDUCACION FLEXIBLE", "EDUCACION FLEXIBLE"),
        # --- Liderazgo: política pública ---
        (r"POLITIC.? PUBLICA DE JUVENTUD.*", "POLITICA PUBLICA DE JUVENTUD"),
        (r"^POLITICA PUBLICA$", "POLITICA PUBLICA DE JUVENTUD"),
        (r"POLITICA PUBLICA JUVENTUD", "POLITICA PUBLICA DE JUVENTUD"),
        (r"PLITICA PUBLICA DE JUVENTUD", "POLITICA PUBLICA DE JUVENTUD"),
        (r"SICIALIZACION DE POLITICA PUBLICA DE JUVENTUD", "POLITICA PUBLICA DE JUVENTUD"),
        (r"POLTICA PUBLICA.*", "POLITICA PUBLICA DE JUVENTUD"),
        (r"LIDERAZGO DE POLITICA PUBLICA JUVENTUD", "POLITICA PUBLICA DE JUVENTUD"),
        (r"USO DE ESPACIOS\s+POLITICA PUBLICA DE JUVENTUD", "POLITICA PUBLICA DE JUVENTUD"),
        (r"^LIDERAZGO DE POLITICA PUBLICA DE JUVENTUD", "POLITICA PUBLICA DE JUVENTUD"),
        # --- Liderazgo juvenil ---
        (r"LIDERE?A?ZGO JUVENIL", "LIDERAZGO JUVENIL"),
        (r"^LIDERAZGO$", "LIDERAZGO JUVENIL"),
        (r"^PARTICIPACION$", "LIDERAZGO JUVENIL"),
        (r"INCIDENCIA PARTICIPACION Y LIDERAZGO", "LIDERAZGO JUVENIL"),
        (r"^Y PRODUCTIVA$", "ORIENTACION SOCIO-OCUPACIONAL"),
        # --- Asesoría jurídica ---
        (r"GESTION CUTURAL", "GESTION CULTURAL"),
        (r"ASES.?RO?I?A JURIDICA Y PARTICIPACI.?N?.*", "ASESORIA JURIDICA Y PARTICIPACION"),
        (r"ASESORIA JURIDICA\b.*", "ASESORIA JURIDICA Y PARTICIPACION"),
        (r"^FUTBOL Y TRANSFORMACION SOCIAL ASESORIAS JURIDICAS", "ASESORIA JURIDICA Y PARTICIPACION"),
    ]:
        t = re.sub(patron, reemplazo, t)
    # Quitar sufijos de detalle
    t = re.split(r"\s+(?:STAND\b|COLEGIO\b|IED\b|ICBF\b|SENA\b|UNAL\b|IDIPRON\b)", t)[0]
    t = t.replace("_", "").replace(".", "").strip()
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["curso_limpio"] = df["NOMBRE_CURSO"].apply(normalizar_curso)
print(f"Cursos normalizados: {df['NOMBRE_CURSO'].nunique()} -> {df['curso_limpio'].nunique()} agrupados")

Cursos normalizados: 666 -> 309 agrupados


## Funciones de formato y estadísticas generales

In [6]:
# Funciones de formato (español colombiano: miles con punto, decimales con coma)
def cap(texto):
    """Mayúscula solo en la primera letra de la frase."""
    if not texto:
        return texto
    t = str(texto).strip().lower()
    siglas = ["spa", "tic", "vbg", "dsdr", "ruv", "lgbtiq+", "srpa", "sirbe", "sdis", "ppdj"]
    for s in siglas:
        t = re.sub(r'\b' + s + r'\b', s.upper(), t, flags=re.IGNORECASE)
    return t[0].upper() + t[1:]

def fmt(n):
    if isinstance(n, float):
        e = int(n); d = round((n - e) * 100)
        return f"{e:,}".replace(",", ".") + f",{d:02d}"
    return f"{n:,}".replace(",", ".")

def pct(n, total):
    return f"{n/total*100:.1f}".replace(".", ",")

def pctv(valor):
    return f"{valor:.1f}".replace(".", ",")

# Estadísticas generales (solo Casas de Juventud)
total_atenciones = len(df)
personas_unicas = df["IDPERSONA"].nunique()
n_localidades = df["LOCALIDAD_ATENCION"].nunique()
cursos_por_persona = round(total_atenciones / personas_unicas, 2)

print(f"Total atenciones: {fmt(total_atenciones)}")
print(f"Personas únicas:  {fmt(personas_unicas)}")
print(f"Localidades:      {n_localidades}")
print(f"Cursos/persona:   {str(cursos_por_persona).replace('.', ',')}")

Total atenciones: 39.387
Personas únicas:  35.880
Localidades:      17
Cursos/persona:   1,1


In [7]:
# Generar sección Equipo desde Excel + imagen de estructura
equipo_excel = os.path.join(BASE, "equipo_casas_juventud.xlsx")

# Colores fijos por subgrupo
COLORES_SUBGRUPO = {
    "Unidades operativas": "#663A93",
    "Estrategia móvil": "#c0392b",
    "SIDICU": "#e67e22",
    "Administrativo": "#663A93",
    "Ejes del servicio (calidad)": "#2c3e50",
}
DESC_SUBGRUPO = {
    "Ejes del servicio (calidad)": "Garantizan la calidad técnica del servicio, procesos de formación, diseño de herramientas pedagógicas y ejecución de eventos.",
}

if os.path.exists(equipo_excel):
    df_equipo = pd.read_excel(equipo_excel)

    equipo_html = '<div class="card">
'
    equipo_html += '                <h2 class="card-title">Equipo</h2>
'
    equipo_html += '                <p style="color:#666; margin-bottom:20px;">Estructura organizacional de Casas de Juventud.</p>
'

    # Imagen de estructura del servicio
    equipo_html += '                <div style="text-align:center; margin:20px 0 30px; background:#fff; border-radius:12px; padding:20px;">
'
    equipo_html += '                    <img src="imagenes/servicios%20sdis%20juventud.png" alt="Estructura del servicio" style="max-width:100%;">
'
    equipo_html += '                </div>
'

    # 1. Coordinadora (desde Excel)
    df_coord = df_equipo[df_equipo["Grupo"] == "Coordinación"]
    if not df_coord.empty:
        coord_nombre = df_coord.iloc[0]["Nombre"]
        coord_cargo = df_coord.iloc[0]["Cargo"]
        equipo_html += f'                <div style="background:linear-gradient(135deg, #2c3e50, #34495e); color:#fff; border-radius:10px; padding:18px 25px; text-align:center; margin-bottom:25px;">
'
        equipo_html += f'                    <div style="font-size:0.75rem; text-transform:uppercase; letter-spacing:1.5px; opacity:0.8; margin-bottom:4px;">1. {coord_cargo}</div>
'
        equipo_html += f'                    <div style="font-size:1.15rem; font-weight:700;">{coord_nombre}</div>
'
        equipo_html += '                </div>
'
        df_equipo = df_equipo[df_equipo["Grupo"] != "Coordinación"]


    # Generar secciones por grupo
    grupo_num = 2
    for grupo in df_equipo["Grupo"].unique():
        sub_grupo = df_equipo[df_equipo["Grupo"] == grupo]
        equipo_html += f'
                <h3 class="card-subtitle">{grupo_num}. {grupo}</h3>
'
        # Grid de tarjetas
        equipo_html += '                <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(250px, 1fr)); gap:16px; margin-bottom:25px;">
'

        sub_num = 1
        for subgrupo in sub_grupo["Subgrupo"].unique():
            personas = sub_grupo[sub_grupo["Subgrupo"] == subgrupo]
            color = COLORES_SUBGRUPO.get(subgrupo, "#663A93")
            desc = DESC_SUBGRUPO.get(subgrupo, "")

            equipo_html += f'                    <div style="border-top:3px solid {color}; border-radius:10px; padding:18px; background:#f8f9fa; box-shadow:0 1px 4px rgba(0,0,0,0.06);">
'
            equipo_html += f'                        <h4 style="font-size:0.9rem; color:{color}; margin:0 0 {"4px" if desc else "12px"};">{grupo_num}.{sub_num} {subgrupo}</h4>
'
            if desc:
                equipo_html += f'                        <p style="font-size:0.8rem; color:#888; margin:0 0 12px;">{desc}</p>
'

            # Personas con nombre
            for _, p in personas.iterrows():
                nombre = str(p["Nombre"]).strip() if pd.notna(p["Nombre"]) and str(p["Nombre"]).strip() else ""
                cargo = str(p["Cargo"]).strip() if pd.notna(p["Cargo"]) else ""
                numero = str(int(p["Numero"])) if pd.notna(p.get("Numero")) and str(p["Numero"]).strip() not in ["", "nan"] else ""
                obs = str(p["Observaciones"]).strip() if pd.notna(p.get("Observaciones")) and str(p["Observaciones"]).strip() not in ["", "nan"] else ""

                if nombre:
                    equipo_html += f'                        <div style="padding:5px 0;"><strong>{nombre}</strong> <span style="color:#888; font-size:0.85rem;">— {cargo}</span>'
                    if obs:
                        equipo_html += f' <span style="color:#aaa; font-size:0.8rem;">({obs})</span>'
                    equipo_html += '</div>
'

            # Conteos (badges)
            badges = []
            for _, p in personas.iterrows():
                nombre = str(p["Nombre"]).strip() if pd.notna(p["Nombre"]) and str(p["Nombre"]).strip() else ""
                if not nombre:
                    cargo = str(p["Cargo"]).strip() if pd.notna(p["Cargo"]) else ""
                    numero = str(int(p["Numero"])) if pd.notna(p.get("Numero")) and str(p["Numero"]).strip() not in ["", "nan"] else ""
                    obs = str(p["Observaciones"]).strip() if pd.notna(p.get("Observaciones")) and str(p["Observaciones"]).strip() not in ["", "nan"] else ""
                    if numero:
                        badge_text = f'<strong>{numero}</strong> {cargo}'
                        if obs:
                            badge_text += f' ({obs})'
                        badges.append(badge_text)

            if badges:
                equipo_html += '                        <div style="margin-top:10px; display:flex; gap:8px; flex-wrap:wrap;">
'
                for b in badges:
                    equipo_html += f'                            <span style="padding:6px 12px; background:#e8ecf1; border-radius:6px; font-size:0.82rem;">{b}</span>
'
                equipo_html += '                        </div>
'

            equipo_html += '                    </div>
'
            sub_num += 1

        equipo_html += '                </div>
'
        grupo_num += 1

    equipo_html += '            </div>'
    print(f"Equipo generado desde Excel: {len(df_equipo)} filas")
else:
    equipo_html = '<div class="card"><h2 class="card-title">Equipo</h2><p>No se encontró equipo_casas_juventud.xlsx</p></div>'
    print("Excel de equipo no encontrado")


In [ ]:
# Generar directorio de Casas de Juventud desde Excel
directorio_excel = os.path.join(BASE, "directorio_casas_juventud.xlsx")

if os.path.exists(directorio_excel):
    df_dir = pd.read_excel(directorio_excel)
    
    directorio_html = '<div style="margin-top:25px;">
'
    directorio_html += '                <h3 class="card-subtitle">Directorio</h3>
'
    directorio_html += '                <table style="width:100%; border-collapse:collapse; font-size:0.85rem;">
'
    directorio_html += '                    <thead>
'
    directorio_html += '                        <tr style="background:#f8f9fa;">
'
    directorio_html += '                            <th style="padding:10px; border-bottom:2px solid #663A93; text-align:left;">Casa de Juventud</th>
'
    directorio_html += '                            <th style="padding:10px; border-bottom:2px solid #663A93; text-align:left;">Localidad</th>
'
    directorio_html += '                            <th style="padding:10px; border-bottom:2px solid #663A93; text-align:left;">Dirección</th>
'
    directorio_html += '                            <th style="padding:10px; border-bottom:2px solid #663A93; text-align:left;">Barrio</th>
'
    directorio_html += '                        </tr>
'
    directorio_html += '                    </thead>
'
    directorio_html += '                    <tbody>
'
    
    for idx, row in df_dir.iterrows():
        bg = '#fff' if idx % 2 == 0 else '#f8f9fa'
        directorio_html += f'                        <tr style="background:{bg};">
'
        directorio_html += f'                            <td style="padding:8px 10px; border-bottom:1px solid #eee;"><strong>{row["Casa de Juventud"]}</strong></td>
'
        directorio_html += f'                            <td style="padding:8px 10px; border-bottom:1px solid #eee;">{row["Localidad"]}</td>
'
        directorio_html += f'                            <td style="padding:8px 10px; border-bottom:1px solid #eee;">{row["Dirección"]}</td>
'
        directorio_html += f'                            <td style="padding:8px 10px; border-bottom:1px solid #eee;">{row["Barrio"]}</td>
'
        directorio_html += '                        </tr>
'
    
    directorio_html += '                    </tbody>
'
    directorio_html += '                </table>
'
    directorio_html += '            </div>'
    print(f"Directorio generado desde Excel: {len(df_dir)} casas")
else:
    directorio_html = '<p style="color:#888;">No se encontró directorio_casas_juventud.xlsx</p>'
    print("Excel de directorio no encontrado")


In [ ]:
# Generar mapa interactivo de Casas de Juventud
import folium
import unicodedata
import re

geojson_path = os.path.join(BASE, "localidades_bogota.geojson")

def normalizar(texto):
    """Normaliza nombres: quita tildes, Ñ->N, prefijos LA/LOS/EL"""
    texto = unicodedata.normalize('NFC', texto).upper().strip()
    texto = texto.replace('Ñ', 'N').replace('ñ', 'n')
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^(LA |LOS |EL )', '', texto)
    return texto

if os.path.exists(directorio_excel) and os.path.exists(geojson_path):
    df_mapa = pd.read_excel(directorio_excel)
    with open(geojson_path, encoding="utf-8") as f:
        localidades_gj = json.load(f)

    locs_con_casa = set(normalizar(l) for l in df_mapa["Localidad"].unique())

    m = folium.Map(location=[4.624, -74.105], zoom_start=11, tiles="CartoDB positron", width="100%", height="100%")

    def style_localidad(feature):
        nombre = normalizar(feature["properties"]["nombre"])
        if nombre in locs_con_casa:
            return {"fillColor": "#d5d5d5", "color": "#663A93", "weight": 1.5, "fillOpacity": 0.35}
        else:
            return {"fillColor": "#ffffff", "color": "#ccc", "weight": 1, "fillOpacity": 0.1}

    folium.GeoJson(localidades_gj, name="Localidades", style_function=style_localidad,
        tooltip=folium.GeoJsonTooltip(fields=["nombre"], aliases=["Localidad:"])).add_to(m)

    for _, row in df_mapa.iterrows():
        if pd.notna(row.get("Latitud")) and pd.notna(row.get("Longitud")):
            popup_html = f'<div style="font-family:Arial; min-width:200px;"><strong style="color:#663A93; font-size:14px;">{row["Casa de Juventud"]}</strong><br><span style="color:#666; font-size:12px;">📍 {row["Localidad"]}</span><br><span style="font-size:11px;">{row["Dirección"]}</span><br><span style="font-size:11px; color:#888;">Barrio: {row["Barrio"]}</span></div>'
            folium.CircleMarker(location=[row["Latitud"], row["Longitud"]], radius=7, color="#663A93",
                fill=True, fill_color="#c0392b", fill_opacity=0.9, weight=2,
                popup=folium.Popup(popup_html, max_width=250), tooltip=row["Casa de Juventud"]).add_to(m)

    m.save(os.path.join(BASE, "mapa_casas_juventud.html"))
    print(f"Mapa generado con {len(df_mapa)} casas, {len(locs_con_casa)} localidades destacadas")
else:
    print("Faltan archivos para generar mapa")


In [8]:
# Modalidades
def limpiar_mod(nom):
    if pd.isna(nom): return "Sin modalidad"
    return cap(re.sub(r"^\d+\s+", "", str(nom)))

df["ml"] = df["NOMMODALIDAD"].apply(limpiar_mod)
modalidades_casas = df.groupby("ml").size().sort_values(ascending=False)

# Localidades
loc_stats = df.groupby("LOCALIDAD_ATENCION").size().sort_values(ascending=False)

# Demografía
sexo = df["SEXO"].value_counts()
edad = df["GRUPO_ETAREO_1DIA_ANUAL_ACTUA"].value_counts()
etnia = df["ETNIA_ACTUAL"].value_counts()
ruv = df["RUV_ACTUAL"].value_counts()
disc = df["SITUACION_DISCAPACIDAD_ACTUAL"].value_counts(dropna=False)

print("Modalidades Casas:")
print(modalidades_casas)
print(f"\nSexo: {dict(sexo)}")
print(f"Edad: {dict(edad)}")

Modalidades Casas:
ml
Estrategia móvil bmt 2938                   30443
Atención inclusiva para jóvenes bmt 2939     8944
dtype: int64

Sexo: {'MUJER': np.int64(21378), 'HOMBRE': np.int64(17987), 'INTERSEXUAL': np.int64(22)}
Edad: {'13 Y 17 AÑOS': np.int64(23098), '18 Y 26 AÑOS': np.int64(14925), '27 Y 59 AÑOS': np.int64(1342), '6 Y 12 AÑOS': np.int64(13), '0 Y 5 AÑOS': np.int64(9)}


## Generar HTML

Se construye el HTML completo: información de cada eje, servicios, localidades, demografía, resumen y brechas.

In [9]:
# Información de cada eje (imágenes locales, descripciones, actuaciones SIRBE)

EJES_INFO = {
    "BIENESTAR": {
        "id": "bienestar",
        "imagen": "imagenes/bienestar.png",
        "desc": "Acciones de prevencion integral que incluyen prevencion de consumo de SPA, prevencion de violencias basadas en genero VBG, derechos sexuales y reproductivos DDSSYR, salud mental y orientacion psicosocial.",
        "actuacion": "Prevencion Integral (464) — Ruta: 12",
    },
    "CULTURA": {
        "id": "cultura",
        "imagen": "imagenes/cultura.png",
        "desc": "Formacion artistica, uso de espacios para practicas artisticas y culturales, deportes, eventos y aprovechamiento del tiempo libre con enfasis en intereses juveniles.",
        "actuacion": "Manejo Adecuado de Tiempo Libre (1003) — Ruta: 10",
    },
    "INCLUSIÓN SOCIAL Y PRODUCTIVA": {
        "id": "inclusion",
        "imagen": "imagenes/inclusion.png",
        "desc": "Formacion para el proyecto de vida, emprendimiento y empleabilidad, educacion financiera, orientacion socio-ocupacional y habilidades TIC para la inclusion laboral.",
        "actuacion": "Formacion para el Proyecto de Vida (1005) — Ruta: 8",
    },
    "LIDERAZGO Y PARTICIPACIÓN": {
        "id": "liderazgo",
        "imagen": "imagenes/liderazgo.png",
        "desc": "Socializacion de politica publica de juventud, asesoria juridica, liderazgo juvenil, derechos humanos, voluntariado intergeneracional y participacion ciudadana.",
        "actuacion": "Asesoria Juridica y Participacion (1004), Politica Publica de Juventud (739), Voluntariado Intergeneracional (1007) — Ruta: 5",
    },
}

ORDEN_EJES = ["BIENESTAR", "CULTURA", "INCLUSIÓN SOCIAL Y PRODUCTIVA", "LIDERAZGO Y PARTICIPACIÓN"]
print("Ejes configurados:", len(EJES_INFO))

Ejes configurados: 4


In [10]:
# Generar quick links que abren los HTMLs externos de cada eje
sidebar_ejes = ""
quick_links = ""

ejes_html = {
    "BIENESTAR": "ejes/Bienestar.html",
    "CULTURA": "ejes/Cultura.html",
    "INCLUSIÓN SOCIAL Y PRODUCTIVA": "ejes/Inclusion.html",
    "LIDERAZGO Y PARTICIPACIÓN": "ejes/Liderazgo.html",
}

for eje in ORDEN_EJES:
    info = EJES_INFO[eje]
    href = ejes_html[eje]
    quick_links += f"""                        <a class="quick-link" href="{href}" target="_blank" style="text-decoration:none; color:inherit;">
                            <div class="quick-link-img"><img src="{info['imagen']}" alt="{cap(eje)}"></div>
                            <div class="quick-link-title">{cap(eje)}</div>
                        </a>\n"""

print("Quick links generados (apuntan a HTMLs externos)")

Quick links generados (apuntan a HTMLs externos)


In [11]:
# Generar secciones auxiliares + datos JSON para filtros interactivos
import json as _json

# --- Modalidades ---
mcH = ""
for m, n in modalidades_casas.items():
    mcH += f'                        <div class="list-item"><div><strong>{m}</strong></div><span class="badge badge-primary">{fmt(n)} atenciones ({pctv(n/total_atenciones*100)}%)</span></div>\n'

ejes_badges = ""
for eje in ORDEN_EJES:
    n_eje = (df["eje"] == eje).sum()
    p = n_eje / total_atenciones * 100
    bc = "badge-success" if p >= 10 else "badge-info"
    ejes_badges += f'                        <span class="badge {bc}">{cap(eje)} ({pctv(p)}%)</span>\n'

# --- Corrección tildes localidades ---
tildes_loc = {
    "Ciudad bolivar": "Ciudad Bol\u00edvar",
    "Engativa": "Engativ\u00e1",
    "Fontibon": "Fontib\u00f3n",
    "San cristobal": "San Crist\u00f3bal",
    "Antonio nari\u00f1o": "Antonio Nari\u00f1o",
    "Rafael uribe": "Rafael Uribe Uribe",
    "Los martires": "Los M\u00e1rtires",
    "Barrios unidos": "Barrios Unidos",
    "Puente aranda": "Puente Aranda",
}

def nombre_loc(loc):
    c = cap(loc)
    return tildes_loc.get(c, c)

# === DATOS JSON PARA FILTROS INTERACTIVOS ===
# Localidades por eje
datos_loc = {}
# Total
loc_total = df.groupby("LOCALIDAD_ATENCION").size().reset_index(name="atenciones").sort_values("atenciones", ascending=False).set_index("LOCALIDAD_ATENCION")
datos_loc["Total"] = [
    {"loc": nombre_loc(l), "at": int(r["atenciones"])}
    for l, r in loc_total.iterrows()
]
# Por eje
for eje in ORDEN_EJES:
    sub = df[df["eje"] == eje]
    loc_eje = sub.groupby("LOCALIDAD_ATENCION").size().reset_index(name="atenciones").sort_values("atenciones", ascending=False).set_index("LOCALIDAD_ATENCION")
    ult_eje = sub.drop_duplicates(subset="IDPERSONA", keep="last")
    pe_eje = ult_eje.groupby("LOCALIDAD_ATENCION").size()
    datos_loc[cap(eje)] = [
        {"loc": nombre_loc(l), "at": int(r["atenciones"])}
        for l, r in loc_eje.iterrows()
    ]

# Demografía por eje (edad, sexo, etnia, ruv, discapacidad)
datos_demo = {}
for filtro in ["Total"] + [cap(e) for e in ORDEN_EJES]:
    sub = df if filtro == "Total" else df[df["eje"] == [e for e in ORDEN_EJES if cap(e) == filtro][0]]
    n = len(sub)
    if n == 0:
        continue
    datos_demo[filtro] = {
        "edad": [{"cat": str(g), "n": int(v)} for g, v in sub["GRUPO_ETAREO_1DIA_ANUAL_ACTUA"].value_counts().items()],
        "sexo": [{"cat": cap(str(g)), "n": int(v)} for g, v in sub["SEXO"].value_counts().items()],
        "etnia": [{"cat": cap(str(g)), "n": int(v)} for g, v in sub["ETNIA_ACTUAL"].value_counts().items()],
        "ruv": [{"cat": "Registrados en RUV" if g == "SI" else "No registrados", "n": int(v)} for g, v in sub["RUV_ACTUAL"].value_counts().items()],
        "disc": [{"cat": cap(str(g)) if pd.notna(g) else "Sin informacion", "n": int(v)} for g, v in sub["SITUACION_DISCAPACIDAD_ACTUAL"].value_counts(dropna=False).items()],
        "total": n,
    }

datos_loc_json = _json.dumps(datos_loc, ensure_ascii=False)
datos_demo_json = _json.dumps(datos_demo, ensure_ascii=False)

# --- Localidades HTML estático (para la tabla inicial) ---
loc_at = df.groupby("LOCALIDAD_ATENCION").size().reset_index(name="atenciones")
loc_df = loc_at.copy()
loc_df.columns = ["localidad", "atenciones"]
loc_df = loc_df.sort_values("atenciones", ascending=False).reset_index(drop=True)
loc_df["pct"] = loc_df["atenciones"] / total_atenciones * 100
loc_df["localidad_fmt"] = loc_df["localidad"].apply(nombre_loc)

t3 = loc_df.head(3)["localidad_fmt"].tolist()
t3p = loc_df.head(3)["pct"].sum()
b3 = loc_df.tail(3)["localidad_fmt"].tolist()

# --- Resumen ---
ejes_resumen = ""
for eje in ORDEN_EJES:
    n = (df["eje"] == eje).sum()
    p = n / total_atenciones * 100
    pers = df[df["eje"] == eje]["IDPERSONA"].nunique()
    ejes_resumen += f'                            <tr><td>{cap(eje)}</td><td>{fmt(n)}</td><td>{pctv(p)}%</td><td>{fmt(pers)}</td></tr>\n'

# --- Brechas ---
sin_clasificar = df[df["eje"] == "SIN CLASIFICAR"]
n_sin = len(sin_clasificar)
pct_sin = n_sin / total_atenciones * 100

sinH = ""
if n_sin > 0:
    for cod, n in sin_clasificar["NOMACTIVIDAD_CURSO"].value_counts().items():
        sinH += f'                            <tr><td>{cod}</td><td>{fmt(n)}</td><td>{pctv(n/n_sin*100)}%</td></tr>\n'

# --- Enfoque diferencial ---
difH = ""
for lab, n, p in [
    ("Personas con discapacidad", (df["SITUACION_DISCAPACIDAD_ACTUAL"]=="SI").sum(), (df["SITUACION_DISCAPACIDAD_ACTUAL"]=="SI").sum()/total_atenciones*100),
    ("Poblacion afrodescendiente", df["ETNIA_ACTUAL"].str.contains("AFRO|NEGRO",case=False,na=False).sum(), df["ETNIA_ACTUAL"].str.contains("AFRO|NEGRO",case=False,na=False).sum()/total_atenciones*100),
    ("Poblacion indigena", df["ETNIA_ACTUAL"].str.contains("INDIGENA",case=False,na=False).sum(), df["ETNIA_ACTUAL"].str.contains("INDIGENA",case=False,na=False).sum()/total_atenciones*100),
    ("Poblacion LGBTIQ+", df["NOMORIENTACION_ACTUAL"].str.contains("HOMO|BISEX|LESB|GAY|PANSES",case=False,na=False).sum(), df["NOMORIENTACION_ACTUAL"].str.contains("HOMO|BISEX|LESB|GAY|PANSES",case=False,na=False).sum()/total_atenciones*100),
]:
    difH += f'                            <tr><td>{lab}</td><td>{fmt(n)}</td><td>{pctv(p)}%</td></tr>\n'

print("Datos JSON generados:")
print(f"  Localidades: {len(datos_loc)} filtros, {len(datos_loc['Total'])} localidades")
print(f"  Demografía: {len(datos_demo)} filtros")
print("Secciones auxiliares generadas")

Datos JSON generados:
  Localidades: 5 filtros, 17 localidades
  Demografía: 5 filtros
Secciones auxiliares generadas


In [12]:
# CSS + ensamblar HTML completo + guardar archivo

CSS = """        @import url('https://fonts.googleapis.com/css2?family=Figtree:wght@400;500;600;700;800&display=swap');
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: 'Figtree', 'Segoe UI', sans-serif; background-color: #ffffff; color: #2F3E3C; }
        .header { background: #2F3E3C; color: #F8F4E1; padding: 20px 30px; display: flex; justify-content: space-between; align-items: center; box-shadow: 0 2px 10px rgba(0,0,0,0.15); }
        .header h1 { font-size: 1.5rem; font-weight: 700; } .header .subtitle { font-size: 0.9rem; opacity: 0.85; }
        .home-btn { font-size: 1.5rem; cursor: pointer; padding: 5px 12px; border-radius: 8px; transition: background 0.2s; } .home-btn:hover { background: rgba(255,255,255,0.15); }
        .container { display: flex; min-height: calc(100vh - 80px); }
        .sidebar { width: 280px; background: #fff; border-right: 1px solid #e0e0e0; padding: 20px 0; overflow-y: auto; }
        .sidebar-section { margin-bottom: 10px; }
        .sidebar-title { padding: 12px 20px; font-weight: 600; color: #663A93; cursor: pointer; display: flex; justify-content: space-between; align-items: center; transition: background 0.2s; }
        .sidebar-title:hover { background: #f5f0fa; } .sidebar-title .arrow { transition: transform 0.2s; } .sidebar-title.active .arrow { transform: rotate(90deg); }
        .sidebar-items { display: none; padding-left: 20px; } .sidebar-items.show { display: block; }
        .sidebar-item { padding: 10px 20px; cursor: pointer; font-size: 0.9rem; color: #3A3A3A; border-left: 3px solid transparent; transition: all 0.2s; }
        .sidebar-item:hover { background: #f5f0fa; border-left-color: #663A93; }
        .sidebar-item.active { background: #ede7f6; border-left-color: #663A93; color: #663A93; font-weight: 600; }
        .main-content { flex: 1; padding: 30px; overflow-y: auto; }
        .content-section { display: none; } .content-section.active { display: block; }
        .card { background: white; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.05); padding: 25px; margin-bottom: 20px; }
        .card-title { font-size: 1.4rem; color: #663A93; margin-bottom: 15px; padding-bottom: 10px; border-bottom: 2px solid #ede7f6; }
        .card-subtitle { font-size: 1.1rem; color: #6C6294; margin: 20px 0 10px 0; }
        .badge { display: inline-block; padding: 5px 12px; border-radius: 20px; font-size: 0.8rem; font-weight: 500; margin-right: 8px; margin-bottom: 8px; }
        .badge-primary { background: #ede7f6; color: #663A93; } .badge-success { background: #e6f9f0; color: #1EAE76; }
        .badge-warning { background: #FFF3E0; color: #F58B53; } .badge-info { background: #e0f4f5; color: #1E9DA3; }
        .stats-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 20px; margin: 20px 0; }
        .stat-card { background: linear-gradient(135deg, #663A93 0%, #8179B0 100%); color: white; padding: 20px; border-radius: 10px; text-align: center; }
        .stat-card.alt { background: linear-gradient(135deg, #1E9DA3 0%, #1EAE76 100%); }
        .stat-card.alt2 { background: linear-gradient(135deg, #F4676E 0%, #F58B53 100%); }
        .stat-number { font-size: 2rem; font-weight: 700; } .stat-label { font-size: 0.9rem; opacity: 0.9; }
        .list-group { border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden; }
        .list-item { padding: 15px 20px; border-bottom: 1px solid #e0e0e0; display: flex; justify-content: space-between; align-items: center; }
        .list-item:last-child { border-bottom: none; } .list-item:hover { background: #faf8f2; }
        .progress-bar { height: 8px; background: #e0e0e0; border-radius: 4px; overflow: hidden; width: 100px; }
        .progress-fill { height: 100%; background: linear-gradient(90deg, #663A93, #F4676E); border-radius: 4px; }
        table { width: 100%; border-collapse: collapse; margin: 15px 0; }
        th, td { padding: 12px 15px; text-align: left; border-bottom: 1px solid #e0e0e0; }
        th { background: #faf8f2; font-weight: 600; color: #663A93; } tr:hover { background: #faf8f2; }
        .methodology-box { background: #f5f0fa; border-left: 4px solid #663A93; padding: 15px 20px; margin: 15px 0; border-radius: 0 8px 8px 0; }
        .materials-list { display: flex; flex-wrap: wrap; gap: 10px; margin: 10px 0; }
        .welcome-section { text-align: center; padding: 60px 20px; }
        .welcome-section h2 { font-size: 2rem; color: #663A93; margin-bottom: 20px; }
        .welcome-section p { font-size: 1.1rem; color: #666; max-width: 600px; margin: 0 auto 30px; }
        .quick-links { display: grid; grid-template-columns: repeat(2, 190px); justify-content: center; gap: 30px; }
        .quick-link { background: white; padding: 25px 20px; border-radius: 16px; box-shadow: 0 2px 12px rgba(0,0,0,0.06); text-align: center; cursor: pointer; transition: transform 0.3s, box-shadow 0.3s; width: 190px; }
        .quick-link:hover { transform: translateY(-6px); box-shadow: 0 8px 25px rgba(102,58,147,0.2); }
        .quick-link-img { width: 120px; height: 120px; margin: 0 auto 15px; border-radius: 50%; overflow: hidden; border: 3px solid #ede7f6; box-shadow: 0 4px 15px rgba(102,58,147,0.15); transition: border-color 0.3s, transform 0.3s; }
        .quick-link:hover .quick-link-img { border-color: #663A93; transform: scale(1.05); }
        .quick-link:hover .sidicu-img { border-color: #663A93; }
        .quick-link-img img { width: 100%; height: 100%; object-fit: cover; }
        .quick-link-title { font-weight: 600; color: #663A93; font-size: 0.95rem; }
        .hbar-row { display: flex; align-items: center; padding: 8px 0; border-bottom: 1px solid #f0f0f0; }
        .hbar-row:last-child { border-bottom: none; }
        .hbar-label { width: 200px; font-size: 0.9rem; color: #3A3A3A; flex-shrink: 0; }
        .hbar-track { flex: 1; height: 22px; background: #f0ecf5; border-radius: 11px; overflow: hidden; margin: 0 15px; }
        .hbar-fill { height: 100%; border-radius: 11px; transition: width 0.5s ease; min-width: 2px; }
        .hbar-value { width: 150px; font-size: 0.85rem; color: #555; text-align: right; flex-shrink: 0; }
        .hbar-pct { color: #999; }
        .hbar-group { background: #fff; border: 1px solid #e0e0e0; border-radius: 10px; padding: 15px 20px; margin: 10px 0; }
        .filter-btns { display: flex; gap: 8px; flex-wrap: wrap; margin: 15px 0 20px; }
        .filter-btn { padding: 8px 18px; border: 2px solid #663A93; border-radius: 25px; background: white; color: #663A93; font-family: inherit; font-size: 0.85rem; font-weight: 600; cursor: pointer; transition: all 0.2s; }
        .filter-btn:hover { background: #f5f0fa; }
        .filter-btn.active { background: #663A93; color: white; }
        /* --- Estructura SIRBE tipo arbol --- */
        .tree-level {{ margin-left: 25px; padding: 8px 0; }}
        .tree-label {{ font-weight: 600; color: var(--accent-text); }}
        .tree-sublabel {{ color: #666; font-size: 0.9rem; margin-left: 10px; }}
        .tree-item {{
            padding: 6px 15px;
            border-left: 2px solid var(--accent-light);
            margin-left: 20px;
            margin-top: 4px;
        }}
        @media (max-width: 768px) { .container { flex-direction: column; } .sidebar { width: 100%; border-right: none; border-bottom: 1px solid #e0e0e0; } .quick-link-img { width: 90px; height: 90px; } }"""

html = f"""<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Gestor de Conocimiento - Casas de Juventud 2025</title>
    <style>
{CSS}
    </style>
</head>
<body>
    <header class="header">
        <div>
            <h1>Gestor de Conocimiento - Casas de Juventud</h1>
            <div class="subtitle">Subdirecci&oacute;n para la Juventud | Casas de Juventud | Datos 2025</div>
        </div>
        <div class="home-btn" onclick="showContent('welcome')" title="Inicio">
            <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#F8F4E1" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M3 9l9-7 9 7v11a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2z"/><polyline points="9 22 9 12 15 12 15 22"/></svg>
        </div>
    </header>
    <div class="container">
        <nav class="sidebar">
            <div class="sidebar-section"><div class="sidebar-title active" onclick="toggleSection(this)"><span>Contexto</span><span class="arrow">&#9654;</span></div>
                <div class="sidebar-items show"><div class="sidebar-item" onclick="showContent('gestion_conocimiento')">Gestión del conocimiento</div><div class="sidebar-item" onclick="showContent('cambios2026')">A tener en cuenta</div><div class="sidebar-item" onclick="showContent('equipo')">Equipo</div><div class="sidebar-item" onclick="showContent('ubicacion')">Ubicación</div></div></div>

            <div class="sidebar-section"><div class="sidebar-title" onclick="toggleSection(this)"><span>Gestión de datos</span><span class="arrow">&#9654;</span></div>
                <div class="sidebar-items">
                    <div class="sidebar-item" onclick="showContent('flujo_datos')">Flujo de gestión de la información</div>
                    <div class="sidebar-item" onclick="showContent('homologacion')">Homologación SIRBE</div>
                </div></div>



            <div class="sidebar-section"><div class="sidebar-title" onclick="toggleSection(this)"><span>Estadísticas</span><span class="arrow">&#9654;</span></div>
                <div class="sidebar-items">
                    <div class="sidebar-item" onclick="showContent('resumen')">Resumen general 2025</div>
                    <div class="sidebar-item" onclick="showContent('localidades')">Localidades 2025</div>
                        </div></div>
        </nav>
        <main class="main-content">
            <div class="content-section active" id="welcome"><div class="welcome-section">
                <h2>Casas de Juventud</h2>
                <p style="text-align:left; max-width:800px; margin:0 auto 30px; font-size:0.95rem; line-height:1.7; color:#444;">Las Casas de Juventud son espacios distritales dirigidos a jóvenes entre 14 y 28 años, concebidos como el eje territorial de la Política Pública Distrital de Juventud (PPDJ). Articulan la oferta de servicios de la administración distrital para garantizar acceso equitativo a oportunidades de desarrollo personal, social y laboral, y funcionan como la principal puerta de entrada al ecosistema de servicios del Distrito.</p>
                
                <p style="text-align:left; max-width:800px; margin:20px auto 25px; font-size:0.95rem; line-height:1.7; color:#444;">Para materializar este propósito y fortalecer los proyectos de vida de los jóvenes, el servicio de Casas de Juventud estructura su oferta integral a través de los siguientes ejes:</p>
                <div class="quick-links">
{quick_links}                </div>

                <div style="margin-top:20px; display:flex; justify-content:center;">
                    <a class="quick-link" href="ejes/SIDICU.html" target="_blank" style="max-width:400px; width:100%; padding:15px 20px; text-decoration:none; color:inherit;">
                        <div class="sidicu-img" style="width:100%; height:180px; border-radius:12px; overflow:hidden; border:3px solid #ede7f6; box-shadow:0 4px 15px rgba(102,58,147,0.15); transition:border-color 0.3s, transform 0.3s; margin-bottom:12px;">
                            <img src="imagenes/sidicu.png" alt="SIDICU" style="width:100%; height:100%; object-fit:cover;">
                        </div>
                        <div class="quick-link-title" style="font-size:1.1rem;">SIDICU — Sistema Distrital del Cuidado</div>
                    </a>
                </div>
            </div></div>

            <div class="content-section" id="gestion_conocimiento">
                <div class="card">
                    <h2 class="card-title">Gestión del conocimiento</h2>
                    <p style="margin-bottom:15px;">Repositorio de documentación de los procesos de los servicios de la Subdirección de Juventud de SDIS.</p>

                    

                    <h3 class="card-subtitle">¿Qué se busca con este repositorio?</h3>
                    <div class="stats-grid" style="grid-template-columns: repeat(2, 1fr);">
                        <div class="stat-card">
                            <div class="stat-number" style="font-size:1.3rem;">QUÉ</div>
                            <div class="stat-label">Consultar qué hace la Subdirección: servicios, categorías, cursos</div>
                        </div>
                        <div class="stat-card alt">
                            <div class="stat-number" style="font-size:1.3rem;">CÓMO</div>
                            <div class="stat-label">Entender cómo se hace: metodologías, paso a paso, materiales</div>
                        </div>
                        <div class="stat-card alt2">
                            <div class="stat-number" style="font-size:1.3rem;">QUIÉN</div>
                            <div class="stat-label">Memoria de quién lo hace: contactos, responsables</div>
                        </div>
                        <div class="stat-card alt3">
                            <div class="stat-number" style="font-size:1.3rem;">MEMORIA</div>
                            <div class="stat-label">Servir como memoria institucional</div>
                        </div>
                    </div>
                </div>
            </div>

            <div class="content-section" id="cambios2026"><div class="card">
                <h2 class="card-title">A tener en cuenta</h2>

                <div style="text-align:center; margin:20px 0 30px;">
                    <img src="imagenes/contexto1.png" alt="Evolución del modelo de servicio" style="max-width:450px; border-radius:12px; box-shadow:0 2px 12px rgba(0,0,0,0.08);">
                </div>

                <h3 class="card-subtitle">Transformación del servicio</h3>
                <p style="line-height:1.7;">Hasta el 2024, el funcionamiento de las Casas de Juventud se pensaba casi exclusivamente para el “aprovechamiento del tiempo libre” y dependía de la gestión autónoma que lograra cada gestor. A partir de 2025, el servicio se transformó y se estructuró en cuatro ejes temáticos principales: Bienestar, Cultura, Inclusión Productiva y Participación.</p>

                <h3 class="card-subtitle">Cambio metodológico 2026</h3>
                <p style="line-height:1.7;">Para 2026, el funcionamiento tuvo un cambio metodológico. La planeación de la oferta pasó a ser bimensual y ahora exige obligatoriamente un diagnóstico territorial. Esto significa que los gestores deben realizar análisis de fuentes secundarias, mapeo de actores, recorridos territoriales y cartografía social para crear un plan de acción que cruce las necesidades reales de los jóvenes con las metas de la política pública. Adicionalmente, se estandarizó el tipo de oferta en todas las casas bajo formatos específicos: experiencias, talleres, ciclos formativos, semilleros y laboratorios.</p>

                <h3 class="card-subtitle">Enrutamiento</h3>
                <p style="line-height:1.7;">Se implementó un nuevo paso llamado “enrutamiento”. Cuando un joven llega a la Casa de Juventud, se le aplica un cuestionario inicial para identificar sus intereses y necesidades puntuales. A partir de allí, se le orienta hacia la oferta institucional adecuada, evitando bombardearlo con información y mejorando su experiencia.</p>

                <h3 class="card-subtitle">Mínimos operativos y regulación de alianzas</h3>
                <p style="line-height:1.7;">El nivel central ahora establece “mínimos operativos” mensuales por cada eje que todas las localidades deben cumplir, adaptándose a momentos específicos del año (por ejemplo, actividades conmemorativas del 8M o inducción a nuevos consejeros). Asimismo, se le dio orden a la figura del voluntariado; ahora quienes deseen impartir talleres deben llenar un formulario, presentar documentos de seguridad y someter su metodología a revisión técnica.</p>

                <h3 class="card-subtitle">Gestión de información</h3>
                <p style="line-height:1.7;">En el reporte de datos es importante entender cómo se diligencia la información. El gestor que registra la ficha y reporta la actividad no siempre es quien la ejecuta. Esto ocurre por tres dinámicas operativas:</p>
                <ul style="margin:10px 0 15px 25px; line-height:1.7;">
                    <li><strong>Aliados y voluntarios:</strong> parte de la oferta es ejecutada por agentes externos. En estos casos, el gestor asume el acompañamiento logístico y la recolección de datos.</li>
                    <li><strong>Gestores transversales:</strong> existe un equipo especializado de gestores culturales que rota por las localidades dictando talleres específicos de su línea artística.</li>
                    <li><strong>Eventos masivos:</strong> en actividades de gran escala pueden participar varios gestores apoyando la logística, y todos terminan registrando fichas a su nombre para el mismo espacio.</li>
                </ul>

                <div style="text-align:center; margin:30px 0 10px;">
                    <img src="imagenes/contexto2.png" alt="Modelo anterior vs nuevo modelo" style="max-width:100%; border-radius:12px; box-shadow:0 2px 12px rgba(0,0,0,0.08);">
                </div>
            </div></div>

            <div class="content-section" id="equipo">{equipo_html}</div>

            <div class="content-section" id="homologacion"><div class="card">
                <h2 class="card-title">Tabla de homologación por eje</h2>
                <p style="color:#666; font-size:0.9rem; margin-bottom:15px;">Así se organiza la información de cada eje en SIRBE y en la ficha física. Fuente: tabla de homologación institucional 2026.</p>

<h3 class="card-subtitle">Estructura oficial en SIRBE</h3>
                <div style="background:#f8f9fa; border-radius:10px; padding:20px 25px; margin-bottom:25px; border:1px solid #e0e0e0;">
                    <p style="font-weight:600; color:#3A3A3A; margin-bottom:8px;">Jerarquía de campos en SIRBE</p>
                    <p style="font-size:0.85rem; color:#666; margin-bottom:12px;">Cada registro en SIRBE tiene estos 5 niveles de desagregación:</p>
                    <div style="font-size:0.88rem; line-height:1.8;">
                        <div><span style="font-family:monospace; background:#ede7f6; padding:2px 6px; border-radius:4px; font-weight:600; font-size:0.8rem;">SERVICIO</span> — Casas de Juventud</div>
                        <div style="margin-left:25px; border-left:2px solid #ddd; padding-left:15px; margin-top:4px;">
                            <div><span style="font-family:monospace; background:#ede7f6; padding:2px 6px; border-radius:4px; font-weight:600; font-size:0.8rem;">NOMMODALIDAD</span> — Atención Inclusiva BMT 2939 / Estrategia Móvil BMT 2938</div>
                            <div style="margin-left:25px; border-left:2px solid #ddd; padding-left:15px; margin-top:4px;">
                                <div><span style="font-family:monospace; background:#ede7f6; padding:2px 6px; border-radius:4px; font-weight:600; font-size:0.8rem;">ACTUACION_INTERV_CURSO</span> — Varía por eje (Prevención Integral, Manejo Adecuado de Tiempo Libre, etc.)</div>
                                <div style="margin-left:25px; border-left:2px solid #ddd; padding-left:15px; margin-top:4px;">
                                    <div><span style="font-family:monospace; background:#ede7f6; padding:2px 6px; border-radius:4px; font-weight:600; font-size:0.8rem;">NOMACTIVIDAD_CURSO</span> — Actividades oficiales del eje (ver tablas abajo)</div>
                                    <div style="margin-left:25px; border-left:2px solid #ddd; padding-left:15px; margin-top:4px;">
                                        <div><span style="font-family:monospace; background:#ede7f6; padding:2px 6px; border-radius:4px; font-weight:600; font-size:0.8rem;">NOMBRE_CURSO</span> — Campo de texto abierto donde se digita el nombre del curso. Debe coincidir con la tabla de homologación, pero al ser manual genera variaciones</div>
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>

                                    <table style="width:100%; border-collapse:collapse; font-size:0.85rem; margin-bottom:20px;">
                        <thead>
                            <tr style="background:#f8f9fa;">
                                <th style="padding:10px; width:16%; border:1px solid #ddd;">Oferta Casas de Juventud 2026</th>
                                <th style="padding:10px; width:18%; border:1px solid #ddd;">Actuación en SIRBE misional</th>
                                <th style="padding:10px; width:14%; border:1px solid #ddd;">Código Ruta de Oportunidades Juveniles</th>
                                <th style="padding:10px; width:26%; border:1px solid #ddd;">Nombre de la actividad</th>
                                <th style="padding:10px; width:26%; border:1px solid #ddd;">Tipo de actividad (ficha física y SIRBE)</th>
                            </tr>
                        </thead>
                        <tbody>
                            <tr>
                                <td rowspan="5" style="padding:10px; border:1px solid #ddd; font-weight:600; vertical-align:middle; text-align:center; background:#fafafa;">BIENESTAR</td>
                                <td rowspan="5" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; background:#fafafa;">PREVENCIÓN INTEGRAL</td>
                                <td rowspan="5" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; font-size:1.2rem; font-weight:700; background:#fafafa;">12</td>
                                <td style="padding:8px 10px; border:1px solid #ddd; text-align:center;">Derechos sexuales y derechos reproductivos</td>
                                <td rowspan="2" style="padding:8px 10px; border:1px solid #ddd; vertical-align:middle;">7. Talleres informativos en prevención (1486 en SIRBE)</td>
                            </tr>
                            <tr>
                                <td style="padding:8px 10px; border:1px solid #ddd; text-align:center;">Prevención de violencias basadas en género</td>
                            </tr>
                            <tr style="background:#fafafa;">
                                <td style="padding:8px 10px; border:1px solid #ddd; text-align:center;">Salud mental</td>
                                <td style="padding:8px 10px; border:1px solid #ddd;">8. Acompañamiento y orientación psicosocial (511 en SIRBE)</td>
                            </tr>
                            <tr>
                                <td style="padding:8px 10px; border:1px solid #ddd; text-align:center;">Prevención de consumo de sustancias psicoactivas SPA</td>
                                <td style="padding:8px 10px; border:1px solid #ddd;">9. Cuidado frente al consumo responsable de SPA (1487 en SIRBE)</td>
                            </tr>
                            <tr style="background:#fafafa;">
                                <td style="padding:8px 10px; border:1px solid #ddd; text-align:center;">Salas de escucha</td>
                                <td style="padding:8px 10px; border:1px solid #ddd;">10. Centros de escucha (1485 en SIRBE)</td>
                            </tr>
                        </tbody>
                    </table>

                    <table style="width:100%; border-collapse:collapse; font-size:0.85rem; margin-bottom:20px;">
                        <thead>
                            <tr style="background:#f8f9fa;">
                                <th style="padding:10px; width:16%; border:1px solid #ddd;">Oferta Casas de Juventud 2026</th>
                                <th style="padding:10px; width:18%; border:1px solid #ddd;">Actuación en SIRBE misional</th>
                                <th style="padding:10px; width:14%; border:1px solid #ddd;">Código Ruta de Oportunidades Juveniles</th>
                                <th style="padding:10px; width:26%; border:1px solid #ddd;">Nombre de la actividad</th>
                                <th style="padding:10px; width:26%; border:1px solid #ddd;">Tipo de actividad (ficha física y SIRBE)</th>
                            </tr>
                        </thead>
                        <tbody>
                            <tr>
                                <td rowspan="11" style="padding:10px; border:1px solid #ddd; font-weight:600; vertical-align:middle; text-align:center; background:#fafafa;">CULTURA</td>
                                <td rowspan="11" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; background:#fafafa;">MANEJO ADECUADO DE TIEMPO LIBRE</td>
                                <td rowspan="11" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; font-size:1.2rem; font-weight:700; background:#fafafa;">10</td>
                                <td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Medio ambiente</td>
                                <td rowspan="9" style="padding:8px 10px; border:1px solid #ddd; vertical-align:middle;">13. Formación artística focalizada (1489 en SIRBE)</td>
                            </tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Deportes</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Artes musicales</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Artes audiovisuales</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Artes plásticas</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Artes escénicas</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Artes circenses</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Artes urbanas</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Gestión cultural</td></tr>
                            <tr style="border-top:2px solid #ccc;">
                                <td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Uso de espacios para desarrollo de prácticas artísticas</td>
                                <td style="padding:8px 10px; border:1px solid #ddd;">11. Aprovechamiento del tiempo libre con énfasis en intereses juveniles (1207 en SIRBE)</td>
                            </tr>
                            <tr style="border-top:2px solid #ccc;">
                                <td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Semana de la juventud</td>
                                <td style="padding:8px 10px; border:1px solid #ddd; background:#fafafa;">15. Semana de la juventud (1491 en SIRBE)</td>
                            </tr>
                        </tbody>
                    </table>

                    <table style="width:100%; border-collapse:collapse; font-size:0.85rem; margin-bottom:20px;">
                        <thead>
                            <tr style="background:#f8f9fa;">
                                <th style="padding:10px; width:16%; border:1px solid #ddd;">Oferta Casas de Juventud 2026</th>
                                <th style="padding:10px; width:18%; border:1px solid #ddd;">Actuación en SIRBE misional</th>
                                <th style="padding:10px; width:14%; border:1px solid #ddd;">Código Ruta de Oportunidades Juveniles</th>
                                <th style="padding:10px; width:26%; border:1px solid #ddd;">Nombre de la actividad</th>
                                <th style="padding:10px; width:26%; border:1px solid #ddd;">Tipo de actividad (ficha física y SIRBE)</th>
                            </tr>
                        </thead>
                        <tbody>
                            <tr>
                                <td rowspan="8" style="padding:10px; border:1px solid #ddd; font-weight:600; vertical-align:middle; text-align:center; background:#fafafa;">INCLUSIÓN SOCIAL Y PRODUCTIVA</td>
                                <td rowspan="8" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; background:#fafafa;">FORMACIÓN PARA EL PROYECTO DE VIDA</td>
                                <td rowspan="8" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; font-size:1.2rem; font-weight:700; background:#fafafa;">8</td>
                                <td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Habilidades informáticas / Oferta TIC</td>
                                <td rowspan="3" style="padding:8px 10px; border:1px solid #ddd; vertical-align:middle;">16. Salas TIC para el fortalecimiento de habilidades y capacidades juveniles (1496 en SIRBE)</td>
                            </tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Idiomas</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Lecto escritura / Matemáticas</td></tr>
                            <tr style="border-top:2px solid #ccc;">
                                <td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Empleo</td>
                                <td rowspan="3" style="padding:8px 10px; border:1px solid #ddd; vertical-align:middle; background:#fafafa;">17. Formación en emprendimiento y empleabilidad (1497 en SIRBE)</td>
                            </tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Emprendimiento</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Fest oportunidades</td></tr>
                            <tr style="border-top:2px solid #ccc;">
                                <td style="padding:6px 10px; border:1px solid #ddd; text-align:center;">Educación financiera</td>
                                <td rowspan="2" style="padding:8px 10px; border:1px solid #ddd; vertical-align:middle;">18. Orientación socio-ocupacional dirigida a jóvenes (1498 en SIRBE)</td>
                            </tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; text-align:center; background:#fafafa;">Formación (rutas de modelo de educación flexible)</td></tr>
                        </tbody>
                    </table>

                    <table style="width:100%; border-collapse:collapse; font-size:0.85rem; margin-bottom:20px;">
                        <thead>
                            <tr style="background:#f8f9fa;">
                                <th style="padding:10px; width:14%; border:1px solid #ddd;">Oferta Casas de Juventud 2026</th>
                                <th style="padding:10px; width:18%; border:1px solid #ddd;">Actuación en SIRBE misional</th>
                                <th style="padding:10px; width:14%; border:1px solid #ddd;">Código Ruta de Oportunidades Juveniles</th>
                                <th style="padding:10px; width:26%; border:1px solid #ddd;">Nombre de la actividad</th>
                                <th style="padding:10px; width:28%; border:1px solid #ddd;">Tipo de actividad (ficha física y SIRBE)</th>
                            </tr>
                        </thead>
                        <tbody>
                            <tr>
                                <td rowspan="7" style="padding:10px; border:1px solid #ddd; font-weight:600; vertical-align:middle; text-align:center; background:#fafafa;">LIDERAZGO Y PARTICIPACIÓN</td>
                                <td rowspan="3" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; background:#fafafa;">Asesoría jurídica y participación</td>
                                <td rowspan="7" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; font-size:1.2rem; font-weight:700; background:#fafafa;">5</td>
                                <td rowspan="3" style="padding:8px 10px; border:1px solid #ddd; vertical-align:middle; text-align:center;">Asesoría jurídica y participación</td>
                                <td style="padding:6px 10px; border:1px solid #ddd;">22. Promoción y fortalecimiento de actividades de organización juvenil (1494 en SIRBE)</td>
                            </tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; background:#fafafa;">23. Escenarios de diálogos intergeneracionales y de saberes que fortalezcan la ciudadanía juvenil (1495 en SIRBE)</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd;">20. Atención y orientación jurídica a jóvenes (1492 en SIRBE)</td></tr>
                            <tr style="border-top:2px solid #ccc;">
                                <td rowspan="3" style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; background:#fafafa;">Política pública de juventud</td>
                                <td rowspan="3" style="padding:8px 10px; border:1px solid #ddd; vertical-align:middle; text-align:center;">Política pública de juventud</td>
                                <td style="padding:6px 10px; border:1px solid #ddd; background:#fafafa;">3. Socialización de política pública de juventud (1502 en SIRBE)</td>
                            </tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd;">4. Derechos humanos y promoción a la participación (1503 en SIRBE)</td></tr>
                            <tr><td style="padding:6px 10px; border:1px solid #ddd; background:#fafafa;">5. Comités operativos locales de juventud (1504 en SIRBE)</td></tr>
                            <tr style="border-top:2px solid #ccc;">
                                <td style="padding:10px; border:1px solid #ddd; vertical-align:middle; text-align:center; background:#fafafa;">Voluntariado intergeneracional</td>
                                <td style="padding:8px 10px; border:1px solid #ddd; text-align:center;">Voluntariado intergeneracional</td>
                                <td style="padding:6px 10px; border:1px solid #ddd;">6. Voluntariado intergeneracional (1505 en SIRBE)</td>
                            </tr>
                        </tbody>
                    </table>

            </div></div>

                        <div class="content-section" id="ubicacion"><div class="card">
                <h2 class="card-title">Ubicación</h2>
                <p style="color:#666; margin-bottom:20px;">Casas de Juventud en Bogotá.</p>
                <div style="text-align:center; margin-bottom:25px;">
                    <img src="imagenes/Casas%20Mapa.jpg" alt="Mapa de Casas de Juventud" style="max-width:100%; border-radius:12px;">
                </div>
                <div style="text-align:center; margin-bottom:25px;">
                    <iframe src="mapa_casas_juventud.html" style="width:100%; height:500px; border:none; border-radius:12px;"></iframe>
                </div>
                {directorio_html}
            </div></div>

            <div class="content-section" id="flujo_datos"><div class="card">
                <h2 class="card-title">Flujo de gestión de la información</h2>
                <p style="color:#666; margin-bottom:20px;">Proceso completo desde la ejecución de una actividad hasta el reporte oficial de metas.</p>

                <div style="display:flex; gap:8px; flex-wrap:wrap; margin-bottom:25px;">
                    <span style="display:inline-block; padding:4px 12px; border-radius:15px; background:#f5f0fa; color:#663A93; font-size:0.8rem; font-weight:600;">Gestor / aliado externo</span>
                    <span style="display:inline-block; padding:4px 12px; border-radius:15px; background:#e8ecf1; color:#3A3A3A; font-size:0.8rem; font-weight:600;">Administrativo / digitador</span>
                    <span style="display:inline-block; padding:4px 12px; border-radius:15px; background:#f0e6fa; color:#4a2a6d; font-size:0.8rem; font-weight:600;">DADE</span>
                    <span style="display:inline-block; padding:4px 12px; border-radius:15px; background:#f0e6fa; color:#4a2a6d; font-size:0.8rem; font-weight:600;">Líder administrativa</span>
                </div>

                <div style="position:relative; padding-left:30px;">
                    <div style="position:absolute; left:12px; top:0; bottom:0; width:3px; background:linear-gradient(to bottom, #663A93, #4a2a6d); border-radius:2px;"></div>

                    <div style="position:relative; margin-bottom:25px;">
                        <div style="position:absolute; left:-24px; top:4px; width:14px; height:14px; background:#663A93; border-radius:50%; border:3px solid #f5f0fa;"></div>
                        <h3 style="font-size:1rem; color:#663A93; margin:0 0 6px;">1. Ejecución de la actividad</h3>
                        <span style="display:inline-block; padding:2px 8px; border-radius:10px; font-size:0.7rem; background:#f5f0fa; color:#663A93; margin-bottom:6px;">Gestor / aliado externo</span>
                        <p style="font-size:0.88rem; color:#555; line-height:1.6; margin:0;">La actividad (taller, charla, evento) es ejecutada por el gestor territorial, un aliado externo, voluntario o equipo transversal. Los gestores culturales, por ejemplo, rotan por las localidades dictando talleres específicos de su línea artística.</p>
                    </div>

                    <div style="position:relative; margin-bottom:25px;">
                        <div style="position:absolute; left:-24px; top:4px; width:14px; height:14px; background:#663A93; border-radius:50%; border:3px solid #f5f0fa;"></div>
                        <h3 style="font-size:1rem; color:#663A93; margin:0 0 6px;">2. Recolección en campo</h3>
                        <span style="display:inline-block; padding:2px 8px; border-radius:10px; font-size:0.7rem; background:#f5f0fa; color:#663A93; margin-bottom:6px;">Gestor / aliado externo</span>
                        <p style="font-size:0.88rem; color:#555; line-height:1.6; margin:0 0 8px;">Se diligencia la <strong>Ficha SIRBE 328</strong> (FOR-PSS-328), un formato físico único con dos secciones:</p>
                        <ul style="font-size:0.88rem; color:#555; line-height:1.7; margin:0 0 0 20px; padding:0;">
                            <li><strong>Sección A (“cabezote”):</strong> la diligencia el gestor. Registra nombre de la actividad, fecha, horas, lugar, modalidad y tipo de actividad.</li>
                            <li><strong>Sección B (listado de asistencia):</strong> la llenan los jóvenes participantes. Registran datos sociodemográficos y responden preguntas transversales de política pública (ej: derechos vulnerados, ruta de oportunidades juveniles). Cada joven firma como constancia de participación.</li>
                        </ul>
                    </div>

                    <div style="position:relative; margin-bottom:25px;">
                        <div style="position:absolute; left:-24px; top:4px; width:14px; height:14px; background:#663A93; border-radius:50%; border:3px solid #f5f0fa;"></div>
                        <h3 style="font-size:1rem; color:#663A93; margin:0 0 6px;">3. Entrega física</h3>
                        <span style="display:inline-block; padding:2px 8px; border-radius:10px; font-size:0.7rem; background:#e8ecf1; color:#3A3A3A; margin-bottom:6px;">Administrativo</span>
                        <p style="font-size:0.88rem; color:#555; line-height:1.6; margin:0;">Los gestores entregan las fichas físicas y listados al administrativo de la Casa de Juventud. <strong>Plazo: días 1 al 20 de cada mes.</strong></p>
                    </div>

                    <div style="position:relative; margin-bottom:25px;">
                        <div style="position:absolute; left:-24px; top:4px; width:14px; height:14px; background:#663A93; border-radius:50%; border:3px solid #f5f0fa;"></div>
                        <h3 style="font-size:1rem; color:#663A93; margin:0 0 6px;">4. Digitación en SIRBE</h3>
                        <span style="display:inline-block; padding:2px 8px; border-radius:10px; font-size:0.7rem; background:#e8ecf1; color:#3A3A3A; margin-bottom:6px;">Administrativo / digitador</span>
                        <p style="font-size:0.88rem; color:#555; line-height:1.6; margin:0 0 8px;">El administrativo traduce la ficha física al sistema SIRBE. <strong>Plazo: días 20 al 28 del mismo mes.</strong></p>
                        <ul style="font-size:0.88rem; color:#555; line-height:1.7; margin:0 0 0 20px; padding:0;">
                            <li>Selecciona modalidad y actuación de listas desplegables.</li>
                            <li>Digita el <strong>nombre del curso</strong> en un campo de texto abierto, usando la <strong>tabla de homologación</strong> como referencia.</li>
                            <li>Carga los jóvenes asistentes. El sistema asigna automáticamente el estado “Atendido Curso”.</li>
                        </ul>
                    </div>

                    <div style="position:relative; margin-bottom:10px;">
                        <div style="position:absolute; left:-24px; top:4px; width:14px; height:14px; background:#4a2a6d; border-radius:50%; border:3px solid #f0e6fa;"></div>
                        <h3 style="font-size:1rem; color:#4a2a6d; margin:0 0 6px;">5. Reporte oficial DADE y desagregación manual</h3>
                        <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:6px;">
                            <span style="display:inline-block; padding:2px 8px; border-radius:10px; font-size:0.7rem; background:#f0e6fa; color:#4a2a6d;">DADE</span>
                            <span style="display:inline-block; padding:2px 8px; border-radius:10px; font-size:0.7rem; background:#f0e6fa; color:#4a2a6d;">Líder administrativa</span>
                        </div>
                        <p style="font-size:0.88rem; color:#555; line-height:1.6; margin:0;">La primera semana del mes siguiente, la DADE (Dirección de Análisis y Diseño Estratégico) emite el reporte oficial consolidado a partir de los datos cargados en el sistema. Debido a que DADE entrega la información de manera general, la líder administrativa debe descargar este reporte y realizar un trabajo manual para desagregar los datos por unidad operativa y por equipo móvil, y así poder evaluar el cumplimiento de las metas internas.</p>
                    </div>
                </div>

                
            
                <div style="text-align:center; margin:25px 0 10px;">
                    <img src="imagenes/SIRBE%20informacion.png" alt="Flujo de información SIRBE" style="max-width:100%; border-radius:12px; box-shadow:0 2px 12px rgba(0,0,0,0.08);">
                </div>
            </div></div>

            <div class="content-section" id="localidades"><div class="card">
                <h2 class="card-title">Cobertura por localidades 2025</h2>
                <div class="filter-btns" id="loc-filters">
                    <button class="filter-btn active" onclick="filtrarLoc('Total')">Total</button>
                    <button class="filter-btn" onclick="filtrarLoc('Bienestar')">Bienestar</button>
                    <button class="filter-btn" onclick="filtrarLoc('Cultura')">Cultura</button>
                    <button class="filter-btn" onclick="filtrarLoc('Inclusión social y productiva')">Inclusión</button>
                    <button class="filter-btn" onclick="filtrarLoc('Liderazgo y participación')">Liderazgo</button>
                </div>
                <div class="methodology-box" style="margin-bottom:15px;"><p>Cada barra muestra el número de atenciones (sesiones registradas) por localidad.</p></div>
                <div id="loc-bars"></div>
                <div class="methodology-box">
                    <p><strong>Mayor concentracion:</strong> {", ".join(t3)} concentran el {int(t3p)}% de las atenciones.</p>
                    <p><strong>Oportunidades:</strong> {", ".join(b3)} presentan menor número de atenciones.</p>
                </div>

                <h3 class="card-subtitle">Nota sobre cobertura territorial</h3>
                <div class="methodology-box">
                    <p>Las 20 localidades de Bogota se cubren a traves de 18 Casas de Juventud. Algunas casas atienden localidades agrupadas:</p>
                    <ul style="margin-left: 20px; margin-top: 8px;">
                        <li><strong>Usme-Sumapaz:</strong> una sola casa cubre ambas localidades</li>
                        <li><strong>Santa Fe-Candelaria:</strong> una sola casa cubre ambas localidades</li>
                        <li><strong>Barrios Unidos-Teusaquillo:</strong> una sola casa cubre ambas localidades</li>
                    </ul>
                    <p style="margin-top: 8px;">En la base SIRBE los registros aparecen bajo estos nombres combinados, por lo cual Sumapaz, Santa Fe y Teusaquillo no tienen registros independientes sino que estan incluidas en su respectiva pareja.</p>
                </div>
            
                <div class="methodology-box" style="margin-top:20px;"><p><strong>Fuente:</strong> Sistema para el Registro de Beneficiarios — SIRBE de la Secretaría Distrital de Integración Social (Réplica Misión). Fecha de consulta: 13/01/2026. Reporte de número de atenciones por localidad.</p></div>
            </div></div>

            <div class="content-section" id="resumen"><div class="card">
                <h2 class="card-title">Resumen general 2025</h2>
                <div class="methodology-box" style="margin-bottom:15px;">
                    <p><strong>Tablero oficial:</strong> <a href="https://app.powerbi.com/view?r=eyJrIjoiMzRiNWRkMDQtNThmNC00Yzk5LThjNTItOWI4MzZkYzYwM2EzIiwidCI6ImIzZTMwODA4LWU5YTgtNGYyYS05YmMxLWE3NjBhZTkxMGNmNSIsImMiOjR9" target="_blank" style="color:#663A93;">Seguimiento técnico (Power BI)</a>. Es el tablero oficial de seguimiento de datos de la Subdirección. Las cifras pueden diferir de las presentadas aquí por varias razones:</p><ul style="margin:8px 0 0 20px; font-size:0.88rem; color:#555; line-height:1.7;"><li><strong>Corte de información:</strong> la fecha de consulta de la base puede ser diferente.</li><li><strong>Métrica utilizada:</strong> atenciones (total de sesiones) vs. personas únicas vs. personas por localidad dan números distintos. Una persona que asistió a 3 sesiones genera 3 atenciones pero es 1 persona.</li><li><strong>Servicios:</strong> si se tomaron en cuenta duplicados de todos los servicios de SDIS, SDIS-Juventud, etc., depende del universo de comparación.</li><li><strong>Personas por localidad:</strong> si una persona fue atendida en Engativá y en Suba, al sumar personas por localidad se cuenta en ambas (doble conteo). El total real de personas es menor que la suma por localidad.</li><li><strong>Duplicados por periodo:</strong> si una misma persona fue a la Casa de Juventud en enero y en diciembre, al quitar duplicados solo queda en uno de los dos meses. Dependiendo del criterio (primera o última aparición) la asignación a mes cambia.</li></ul><p style="margin-top:8px; font-size:0.85rem; color:#888;">En este gestor usamos número de atenciones para evidenciar el volumen operativo en cada categoría de análisis.</p><p style="margin-top:8px; font-size:0.85rem; color:#555;">Al solicitar información de SIRBE se debe dejar nota de qué criterios se tomaron en cuenta en el reporte.</p>
                </div>
                <h3 class="card-subtitle">Modalidades</h3>
                <div class="list-group">
{mcH}                </div>
                <h3 class="card-subtitle">Distribucion por eje</h3>
                <table><thead><tr><th>Eje</th><th>Atenciones</th><th>%</th><th>Personas</th></tr></thead><tbody>
{ejes_resumen}                </tbody></table>
                <div class="methodology-box">
                    <p><strong>Servicio:</strong> Casas de Juventud</p>
                    <p><strong>Ano:</strong> 2025 ({fmt(total_atenciones)} registros)</p>
                    <p><strong>Fuente:</strong> SIRBE</p>
                    <p><strong>Generado:</strong> {datetime.now().strftime("%Y-%m-%d")}</p>
                </div>
            
                <div class="methodology-box" style="margin-top:20px;"><p><strong>Fuente:</strong> Sistema para el Registro de Beneficiarios — SIRBE de la Secretaría Distrital de Integración Social (Réplica Misión). Fecha de consulta: 13/01/2026.</p></div>
            </div></div>

                </main>
    </div>
    <script>
        function toggleSection(el){{el.classList.toggle('active');el.nextElementSibling.classList.toggle('show');}}
        function showContent(id){{document.querySelectorAll('.content-section').forEach(s=>s.classList.remove('active'));document.querySelectorAll('.sidebar-item').forEach(i=>i.classList.remove('active'));var s=document.getElementById(id);if(s)s.classList.add('active');if(event&&event.target)event.target.closest('.sidebar-item')?.classList.add('active');}}

        var DL={datos_loc_json};
        var DD={datos_demo_json};

        function fmtN(n){{return n.toLocaleString('es-CO');}}

        function setActive(cid,btn){{document.querySelectorAll('#'+cid+' .filter-btn').forEach(function(b){{b.classList.remove('active')}});if(btn)btn.classList.add('active');}}

        function filtrarLoc(eje){{
            var d=DL[eje];if(!d)return;
            var total=0;d.forEach(function(r){{total+=r.at}});
            var mx=d[0].at;
            var h='';
            d.forEach(function(r){{
                var w=(r.at/mx*100).toFixed(0);
                h+='<div class="hbar-row"><div class="hbar-label">'+r.loc+'</div>'
                  +'<div class="hbar-track"><div class="hbar-fill" style="width:'+w+'%;background:#663A93;"></div></div>'
                  +'<div class="hbar-value">'+fmtN(r.at)+' atenciones</div></div>';
            }});
            h+='<div class="hbar-row" style="font-weight:700;background:#f5f0fa;border-radius:8px;margin-top:5px;">'
              +'<div class="hbar-label">Total Bogot\u00e1</div><div class="hbar-track"></div>'
              +'<div class="hbar-value">'+fmtN(total)+' atenciones</div></div>';
            document.getElementById('loc-bars').innerHTML=h;
            if(event&&event.target)setActive('loc-filters',event.target);
        }}

        function filtrarDemo(eje){{
            var d=DD[eje];if(!d)return;
            var t=d.total;
            ['edad','sexo','etnia','ruv','disc'].forEach(function(cat){{
                var el=document.getElementById('demo-'+cat);
                if(!el||!d[cat])return;
                var h='';
                d[cat].forEach(function(r){{
                    var p=(r.n/t*100);
                    h+='<div class="hbar-row"><div class="hbar-label">'+r.cat+'</div>'
                      +'<div class="hbar-track"><div class="hbar-fill" style="width:'+p.toFixed(1)+'%;background:#663A93;"></div></div>'
                      +'<div class="hbar-value">'+fmtN(r.n)+' <span class="hbar-pct">('+p.toFixed(1).replace('.',',')+'%)</span></div></div>';
                }});
                el.innerHTML=h;
            }});
            if(event&&event.target)setActive('demo-filters',event.target);
        }}

        document.addEventListener('DOMContentLoaded',function(){{filtrarLoc('Total');filtrarDemo('Total')}});
    </script>
</body>
</html>"""

# Guardar
salida = os.path.join(BASE, "gestion_conocimiento_juventud_2025.html")
with open(salida, "w", encoding="utf-8") as f:
    f.write(html)

print(f"HTML generado: {salida}")
print(f"Tamano: {len(html):,} caracteres".replace(",", "."))
print(f"Casas de Juventud 2025: {fmt(total_atenciones)} atenciones, {fmt(personas_unicas)} personas unicas")

HTML generado: C:\Users\carol\CH_projects\SDIS\Gestion de conocimiento\gestor-conocimiento-2025\gestion_conocimiento_juventud_2025.html
Tamano: 65.402 caracteres
Casas de Juventud 2025: 39.387 atenciones, 35.880 personas unicas
